<a href="https://colab.research.google.com/github/viethaa/f1-big-data-analytics/blob/main/formula1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><b>A 2025 Analysis of Formula 1 Top Constructors' Performance and Strategy </b></h1>

Formula 1 is one of the most technologically advanced and data-driven sports in the world. Casual watchers may think that the fastest car is the only determinant factor to win the race, but that is entirely wrong. Across a season of 24 races held on circuits with vastly different layouts and conditions, teams compete through a cummulative effort between the engineers, the pit crew, the strategists, the data analysts, and the drivers.

Every race involves constant strategic decision-making. Teams must manage tyre degradation, fuel load, pit stop timing, overtaking opportunities, track position, weather changes, safety cars, and car setup while adapting to different circuit characteristics. Some circuits reward straight-line speed and low drag efficiency, while others require strong cornering performance and tyre management. Even small differences in pit stop execution or race strategy can determine whether a driver gains or loses positions during a race. Because of this complexity, Formula 1 has become increasingly dependent on large-scale performance analytics and telemetry data to improve decision-making and optimise car performance.

The purpose of this paper is to analyse and compare the 2025 season performances of the 4 leading constructors: Ferrari, McLaren, Mercedes, and Red Bull. The team's performances was analyze using race results, lap timing data, tyre performance metrics, pit stop analysis, and circuit-specific comparisons. Rather than focusing only on championship standings or race wins, the investigation examines how different teams achieved performance across multiple technical and strategic areas throughout the season.

The investigation combines publicly available Formula 1 datasets from the OpenF1 and Jolpica (Ergast) APIs. OpenF1 provided detailed telemetry-style race information such as lap times, tyre compounds, pit stop data, speed trap measurements, and session information, while Ergast provided official race classification data including finishing positions, grid positions, race schedules, and constructor information. These datasets were processed and analysed using Python libraries including Pandas, NumPy, SciPy, and Plotly to build visualisations and statistical comparisons.

The investigation itself is divided into the following sections. Section I focuses on data configuration, preprocessing, API integration, and the preparation of datasets used throughout the investigation. This includes cleaning and organising telemetry-style data, race classifications, lap times, and team mappings required for the later analyses. Sections II examines tyre compounds, stint behaviour, and tyre degradation across soft, medium, and hard tyres. Section III then focuses on pit stop efficiency by analysing pit lane durations, multi-stop race strategies, and how pit stops influenced track position during races. Section IV looks at race position performance by comparing qualifying positions, finishing results, and front-grid position conversion. Section V examines circuit performance characteristics across the season. This includes comparing straight-line speed, speed trap performance, and average finishing results on corner-heavy and straight-heavy circuits in order to identify how different car characteristics influenced performance at different track types. Finally, sections VI and VII discusses limitations, future works, and conclusions. The goal of this paper is to provide a broader understanding of the factors that influenced the 2025 Formula 1 season.
#

Project GitHub: https://github.com/viethaa/f1-big-data-analytics
#
Author Contact: viet.ha@concordiahanoi.org.

# I: Data Configuration

In [ ]:
# Install packages
!pip install -q plotly

In [ ]:
# Import libraries
import os
import requests
import time
import base64
from io import BytesIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from PIL import Image
from google.colab import drive

In [ ]:
# Mount Drive
drive.mount('/content/drive')

YEAR = 2025
DATA_DIR = f"/content/drive/MyDrive/Big Data 2 - Ha, Bach Viet (Bob)/f1_data_{YEAR}"
JOLPICA = "https://api.jolpi.ca/ergast/f1"
OPENF1 = "https://api.openf1.org/v1"
MARKER = os.path.join(DATA_DIR, ".download_complete")
os.makedirs(DATA_DIR, exist_ok=True)

TEAM_COLORS = {
    "Ferrari": "#E8002D",
    "McLaren": "#FF8000",
    "Mercedes": "#27F4D2",
    "Red Bull": "#3671C6",
}

COMPOUND_COLORS = {
    "SOFT": "rgb(255, 45, 45)",
    "MEDIUM": "rgb(255, 210, 50)",
    "HARD": "rgb(230, 230, 230)",
    "INTERMEDIATE": "rgb(60, 180, 60)",
    "WET": "rgb(50, 130, 255)",
}

Mounted at /content/drive


In [ ]:
# Download all data (skips files that already exist)

def api_get(url, params=None):
    for attempt in range(3):
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except:
            time.sleep(2)
    return None

def already_done(filename):
    path = f"{DATA_DIR}/{filename}"
    if os.path.exists(path):
        rows = len(pd.read_csv(path))
        print(f"   Already exists ({rows} rows) — skipping")
        return True
    return False

races = api_get(f"{JOLPICA}/{YEAR}.json")["MRData"]["RaceTable"]["Races"]

# 1. Schedule
print("1. Schedule...")
if not already_done(f"schedule_{YEAR}.csv"):
    rows = [{"round": int(r["round"]), "raceName": r["raceName"],
             "circuitId": r["Circuit"]["circuitId"], "circuitName": r["Circuit"]["circuitName"],
             "country": r["Circuit"]["Location"]["country"], "locality": r["Circuit"]["Location"]["locality"],
             "lat": r["Circuit"]["Location"]["lat"], "long": r["Circuit"]["Location"]["long"],
             "date": r["date"], "time": r.get("time", "")} for r in races]
    pd.DataFrame(rows).to_csv(f"{DATA_DIR}/schedule_{YEAR}.csv", index=False)
    print(f"   Done: {len(rows)} races")
time.sleep(0.5)

# 2. Driver standings
print("2. Driver standings...")
if not already_done(f"driver_standings_{YEAR}.csv"):
    ds = api_get(f"{JOLPICA}/{YEAR}/driverStandings.json")
    sl = ds["MRData"]["StandingsTable"]["StandingsLists"]
    if sl:
        rows = [{"position": int(s["position"]), "driverId": s["Driver"]["driverId"],
                 "driverCode": s["Driver"].get("code",""), "givenName": s["Driver"]["givenName"],
                 "familyName": s["Driver"]["familyName"], "nationality": s["Driver"]["nationality"],
                 "constructorId": s["Constructors"][0]["constructorId"] if s["Constructors"] else "",
                 "constructorName": s["Constructors"][0]["name"] if s["Constructors"] else "",
                 "points": float(s["points"]), "wins": int(s["wins"])} for s in sl[0]["DriverStandings"]]
        pd.DataFrame(rows).to_csv(f"{DATA_DIR}/driver_standings_{YEAR}.csv", index=False)
        print(f"   Done: {len(rows)} drivers")
time.sleep(0.5)

# 3. Constructor standings
print("3. Constructor standings...")
if not already_done(f"constructor_standings_{YEAR}.csv"):
    cs = api_get(f"{JOLPICA}/{YEAR}/constructorStandings.json")
    cl = cs["MRData"]["StandingsTable"]["StandingsLists"]
    if cl:
        rows = [{"position": int(s["position"]), "constructorId": s["Constructor"]["constructorId"],
                 "constructorName": s["Constructor"]["name"], "nationality": s["Constructor"]["nationality"],
                 "points": float(s["points"]), "wins": int(s["wins"])} for s in cl[0]["ConstructorStandings"]]
        pd.DataFrame(rows).to_csv(f"{DATA_DIR}/constructor_standings_{YEAR}.csv", index=False)
        print(f"   Done: {len(rows)} constructors")
time.sleep(0.5)

# 4. Race results
print("4. Race results...")
if not already_done(f"results_{YEAR}.csv"):
    all_results = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/results.json", params={"limit": 30})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "Results" in rd[0]:
                for r in rd[0]["Results"]:
                    fl = r.get("FastestLap") or {}
                    all_results.append({
                        "round": int(rnd), "race_name": name,
                        "driverId": r["Driver"]["driverId"], "driverCode": r["Driver"].get("code",""),
                        "constructorId": r["Constructor"]["constructorId"], "constructorName": r["Constructor"]["name"],
                        "grid": int(r["grid"]), "position": r.get("position"),
                        "positionText": r.get("positionText",""), "status": r.get("status",""),
                        "points": float(r.get("points",0)), "laps": int(r.get("laps",0)),
                        "time_text": r.get("Time",{}).get("time","") if isinstance(r.get("Time"),dict) else "",
                        "fastest_lap_rank": fl.get("rank",""),
                        "fastest_lap_time": fl.get("Time",{}).get("time","") if fl.get("Time") else "",
                        "fastest_lap_speed_kph": fl.get("AverageSpeed",{}).get("speed","") if fl.get("AverageSpeed") else "",
                    })
        except: pass
        time.sleep(0.5)
    pd.DataFrame(all_results).to_csv(f"{DATA_DIR}/results_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_results)}")
time.sleep(0.5)

# 5. Qualifying
print("5. Qualifying...")
if not already_done(f"qualifying_{YEAR}.csv"):
    all_quali = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/qualifying.json", params={"limit": 30})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "QualifyingResults" in rd[0]:
                for q in rd[0]["QualifyingResults"]:
                    all_quali.append({
                        "round": int(rnd), "race_name": name,
                        "driverId": q["Driver"]["driverId"], "driverCode": q["Driver"].get("code",""),
                        "constructorId": q["Constructor"]["constructorId"],
                        "position": int(q["position"]), "Q1": q.get("Q1",""), "Q2": q.get("Q2",""), "Q3": q.get("Q3",""),
                    })
        except: pass
        time.sleep(0.5)
    pd.DataFrame(all_quali).to_csv(f"{DATA_DIR}/qualifying_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_quali)}")
time.sleep(0.5)

# 6. Pit stops
print("6. Pit stops...")
if not already_done(f"pitstops_{YEAR}.csv"):
    all_pits = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/pitstops.json", params={"limit": 200})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "PitStops" in rd[0]:
                for s in rd[0]["PitStops"]:
                    dur = s.get("duration","")
                    sec = float(dur.split(":")[0])*60 + float(dur.split(":")[1]) if ":" in str(dur) else (float(dur) if dur else None)
                    all_pits.append({"round": int(rnd), "race_name": name, "driverId": s["driverId"],
                                     "stop": int(s["stop"]), "lap": int(s["lap"]),
                                     "time_of_day": s.get("time",""), "duration_raw": dur, "duration_sec": sec})
        except: pass
        time.sleep(0.5)
    pd.DataFrame(all_pits).to_csv(f"{DATA_DIR}/pitstops_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_pits)}")
time.sleep(0.5)

# 7. Lap times
print("7. Lap times (slowest step — ~5 min)...")
if not already_done(f"laptimes_{YEAR}.csv"):
    all_laps = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        print(f"   [R{rnd}] {name}...", end=" ", flush=True)
        count = 0
        try:
            offset = 0
            while True:
                resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/laps.json", params={"limit": 1000, "offset": offset})
                rd = resp["MRData"]["RaceTable"]["Races"]
                if not rd or "Laps" not in rd[0] or not rd[0]["Laps"]: break
                for lap in rd[0]["Laps"]:
                    for t in lap["Timings"]:
                        lt = t.get("time","")
                        lt_sec = float(lt.split(":")[0])*60 + float(lt.split(":")[1]) if lt and ":" in lt else None
                        all_laps.append({"round": int(rnd), "race_name": name, "driverId": t["driverId"],
                                         "lap": int(lap["number"]), "position": int(t.get("position",0)),
                                         "time_text": lt, "time_sec": lt_sec})
                        count += 1
                offset += 1000
                if offset >= int(resp["MRData"]["total"]): break
                time.sleep(0.3)
            print(f"done ({count})")
        except: print(f"failed ({count} collected)")
        time.sleep(0.5)
    pd.DataFrame(all_laps).to_csv(f"{DATA_DIR}/laptimes_{YEAR}.csv", index=False)
    print(f"   Total: {len(all_laps)}")
time.sleep(0.5)

# 8. Sprints
print("8. Sprints...")
if not already_done(f"sprints_{YEAR}.csv"):
    all_sprints = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/sprint.json", params={"limit": 30})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "SprintResults" in rd[0]:
                for r in rd[0]["SprintResults"]:
                    all_sprints.append({"round": int(rnd), "race_name": name, "driverId": r["Driver"]["driverId"],
                                        "driverCode": r["Driver"].get("code",""), "constructorId": r["Constructor"]["constructorId"],
                                        "grid": int(r["grid"]), "position": r.get("position"),
                                        "points": float(r.get("points",0)), "laps": int(r.get("laps",0))})
        except: pass
        time.sleep(0.3)
    if all_sprints: pd.DataFrame(all_sprints).to_csv(f"{DATA_DIR}/sprints_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_sprints)}")
time.sleep(0.5)

# 9. OpenF1 stints + drivers
print("9. OpenF1 stints and drivers...")
stints_done = already_done(f"stints_openf1_{YEAR}.csv")
drivers_done = already_done(f"drivers_openf1_{YEAR}.csv")
if not stints_done or not drivers_done:
    sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
    time.sleep(1)
    all_drivers, all_stints = [], []
    for s in sessions:
        sk, name = s["session_key"], s.get("meeting_name", "")
        try:
            d = requests.get(f"{OPENF1}/drivers", params={"session_key": sk}).json()
            for drv in d: all_drivers.append({**drv, "session_key": sk, "race_name": name})
        except: pass
        time.sleep(1)
        for attempt in range(3):
            try:
                resp = requests.get(f"{OPENF1}/stints", params={"session_key": sk}).json()
                if isinstance(resp, list) and len(resp) > 0:
                    for st in resp: all_stints.append({**st, "session_key": sk, "race_name": name})
                    print(f"   [{sk}] {name} done ({len(resp)} stints)")
                    break
            except: pass
            time.sleep(2)
        time.sleep(1.5)
    pd.DataFrame(all_drivers).to_csv(f"{DATA_DIR}/drivers_openf1_{YEAR}.csv", index=False)
    pd.DataFrame(all_stints).to_csv(f"{DATA_DIR}/stints_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_drivers)} drivers, {len(all_stints)} stints")

# 10. OpenF1 pit stops (stationary time + timestamps)
print("10. OpenF1 pit stops (timestamps + stationary time)...")
if not already_done(f"pitstops_openf1_{YEAR}.csv"):
    if 'sessions' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_pits = []
    for s in race_only:
        sk, name = s["session_key"], s.get("circuit_short_name", s.get("meeting_name", ""))
        try:
            resp = requests.get(f"{OPENF1}/pit", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for p in resp:
                    all_of1_pits.append({
                        "session_key": sk,
                        "race_name": name,
                        "driver_number": int(p["driver_number"]),
                        "lap_number": int(p["lap_number"]),
                        "pit_duration": p.get("pit_duration"),
                        "date": p.get("date", ""),
                    })
                print(f"   [{sk}] {name}: {len(resp)} pit stops")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_of1_pits).to_csv(f"{DATA_DIR}/pitstops_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_of1_pits)} pit records")
time.sleep(0.5)

# 11. OpenF1 position data (race position changes with timestamps)
print("11. OpenF1 position data (position timeline)...")
if not already_done(f"positions_openf1_{YEAR}.csv"):
    if 'sessions' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_pos = []
    for s in race_only:
        sk, name = s["session_key"], s.get("circuit_short_name", s.get("meeting_name", ""))
        try:
            resp = requests.get(f"{OPENF1}/position", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for p in resp:
                    all_of1_pos.append({
                        "session_key": sk,
                        "driver_number": int(p["driver_number"]),
                        "position": int(p["position"]),
                        "date": p.get("date", ""),
                    })
                print(f"   [{sk}] {name}: {len(resp)} position records")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_of1_pos).to_csv(f"{DATA_DIR}/positions_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_of1_pos)} position records")
time.sleep(0.5)

# 12. OpenF1 lap durations (for tyre degradation analysis)
print("12. OpenF1 lap durations...")
if not already_done(f"laps_openf1_{YEAR}.csv"):
    if 'sessions' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_laps = []
    for s in race_only:
        sk, name = s["session_key"], s.get("circuit_short_name", s.get("meeting_name", ""))
        try:
            resp = requests.get(f"{OPENF1}/laps", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for lap in resp:
                    if lap.get("lap_duration") and lap["lap_duration"] > 0:
                        all_of1_laps.append({
                            "session_key": int(sk),
                            "driver_number": int(lap["driver_number"]),
                            "lap_number": int(lap["lap_number"]),
                            "lap_duration": float(lap["lap_duration"]),
                        })
                print(f"   [{sk}] {name}: {len([l for l in resp if l.get('lap_duration')])} laps")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_of1_laps).to_csv(f"{DATA_DIR}/laps_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_of1_laps)} lap records")
time.sleep(0.5)

# 13. OpenF1 enriched laps (sector times + speed traps)
print("13. OpenF1 enriched laps (sectors + speed traps)...")
if not already_done(f"laps_enriched_openf1_{YEAR}.csv"):
    if 'sessions' not in dir() or 'race_only' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
        race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_enriched = []
    for s in race_only:
        sk = s["session_key"]
        name = s.get("circuit_short_name", s.get("meeting_name", ""))
        meeting = s.get("meeting_name", "")
        try:
            resp = requests.get(f"{OPENF1}/laps", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for lap in resp:
                    all_enriched.append({
                        "session_key": int(sk),
                        "circuit_short_name": name,
                        "meeting_name": meeting,
                        "driver_number": int(lap["driver_number"]),
                        "lap_number": int(lap.get("lap_number", 0)),
                        "lap_duration": lap.get("lap_duration"),
                        "duration_sector_1": lap.get("duration_sector_1"),
                        "duration_sector_2": lap.get("duration_sector_2"),
                        "duration_sector_3": lap.get("duration_sector_3"),
                        "i1_speed": lap.get("i1_speed"),
                        "i2_speed": lap.get("i2_speed"),
                        "st_speed": lap.get("st_speed"),
                        "segments_sector_1": str(lap.get("segments_sector_1", "")),
                        "segments_sector_2": str(lap.get("segments_sector_2", "")),
                        "segments_sector_3": str(lap.get("segments_sector_3", "")),
                        "is_pit_out_lap": lap.get("is_pit_out_lap"),
                    })
                print(f"   [{sk}] {name}: {len(resp)} laps")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_enriched).to_csv(f"{DATA_DIR}/laps_enriched_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_enriched)} enriched lap records")
time.sleep(0.5)

with open(MARKER, "w") as f: f.write("done")
print(f"\nAll data saved to Google Drive at:\n  {DATA_DIR}/")

1. Schedule...
   Already exists (24 rows) — skipping
2. Driver standings...
   Already exists (21 rows) — skipping
3. Constructor standings...
   Already exists (10 rows) — skipping
4. Race results...
   Already exists (479 rows) — skipping
5. Qualifying...
   Already exists (479 rows) — skipping
6. Pit stops...
   Already exists (821 rows) — skipping
7. Lap times (slowest step — ~5 min)...
   Already exists (3858 rows) — skipping
8. Sprints...
   Already exists (120 rows) — skipping
9. OpenF1 stints and drivers...
   Already exists (1433 rows) — skipping
   Already exists (579 rows) — skipping
10. OpenF1 pit stops (timestamps + stationary time)...
   Already exists (821 rows) — skipping
11. OpenF1 position data (position timeline)...
   Already exists (11564 rows) — skipping
12. OpenF1 lap durations...
   Already exists (26211 rows) — skipping
13. OpenF1 enriched laps (sectors + speed traps)...
   Already exists (26265 rows) — skipping

All data saved to Google Drive at:
  /content/d

In [ ]:
# Load all DataFrames
schedule_df  = pd.read_csv(f"{DATA_DIR}/schedule_{YEAR}.csv")
results_df   = pd.read_csv(f"{DATA_DIR}/results_{YEAR}.csv")
quali_df     = pd.read_csv(f"{DATA_DIR}/qualifying_{YEAR}.csv")
pitstops_df  = pd.read_csv(f"{DATA_DIR}/pitstops_{YEAR}.csv")
laptimes_df  = pd.read_csv(f"{DATA_DIR}/laptimes_{YEAR}.csv")
drv_stand_df = pd.read_csv(f"{DATA_DIR}/driver_standings_{YEAR}.csv")
con_stand_df = pd.read_csv(f"{DATA_DIR}/constructor_standings_{YEAR}.csv")
stints_df    = pd.read_csv(f"{DATA_DIR}/stints_openf1_{YEAR}.csv")
drivers_df   = pd.read_csv(f"{DATA_DIR}/drivers_openf1_{YEAR}.csv")
sprints_path = f"{DATA_DIR}/sprints_{YEAR}.csv"
sprints_df   = pd.read_csv(sprints_path) if os.path.exists(sprints_path) else pd.DataFrame()

of1_pits_df  = pd.read_csv(f"{DATA_DIR}/pitstops_openf1_{YEAR}.csv")
of1_pos_df   = pd.read_csv(f"{DATA_DIR}/positions_openf1_{YEAR}.csv")
of1_laps_df  = pd.read_csv(f"{DATA_DIR}/laps_openf1_{YEAR}.csv")

enriched_path = f"{DATA_DIR}/laps_enriched_openf1_{YEAR}.csv"
laps_enriched_df = pd.read_csv(enriched_path) if os.path.exists(enriched_path) else pd.DataFrame()

print(f"All DataFrames loaded from: {DATA_DIR}/")
for name, df in [("Schedule", schedule_df), ("Results", results_df), ("Qualifying", quali_df),
                 ("Pit Stops", pitstops_df), ("Lap Times", laptimes_df), ("Stints", stints_df),
                 ("Drivers", drivers_df), ("Sprints", sprints_df),
                 ("OpenF1 Pits", of1_pits_df), ("OpenF1 Positions", of1_pos_df), ("OpenF1 Laps", of1_laps_df),
                 ("Enriched Laps", laps_enriched_df)]:
    print(f"  {name:20s} - {len(df)} rows")

All DataFrames loaded from: /content/drive/MyDrive/Big Data 2 - Ha, Bach Viet (Bob)/f1_data_2025/
  Schedule             - 24 rows
  Results              - 479 rows
  Qualifying           - 479 rows
  Pit Stops            - 821 rows
  Lap Times            - 3858 rows
  Stints               - 1433 rows
  Drivers              - 579 rows
  Sprints              - 120 rows
  OpenF1 Pits          - 821 rows
  OpenF1 Positions     - 11564 rows
  OpenF1 Laps          - 26211 rows
  Enriched Laps        - 26265 rows


In [ ]:
# Prepare stint data for top 4 teams
top_teams = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

stints = stints_df.copy()
drivers_lookup = drivers_df[["driver_number", "name_acronym", "team_name", "session_key"]].drop_duplicates()
stints = stints.merge(drivers_lookup, on=["driver_number", "session_key"], how="left")

team_map = {}
for tn in stints["team_name"].dropna().unique():
    for top in top_teams:
        if top.lower() in tn.lower():
            team_map[tn] = top
            break

stints["team_clean"] = stints["team_name"].map(team_map)
stints["stint_length"] = stints["lap_end"] - stints["lap_start"] + 1
top4 = stints[stints["team_clean"].isin(top_teams) & (stints["stint_length"] >= 3)].copy()

print(f"Stint data ready: {len(top4)} stints across {top4['race_name'].nunique()} races")
for team in top_teams:
    t = top4[top4["team_clean"] == team]
    print(f"  {team:12s}: {len(t):3d} stints | "
          f"S:{len(t[t['compound']=='SOFT']):2d}  M:{len(t[t['compound']=='MEDIUM']):2d}  "
          f"H:{len(t[t['compound']=='HARD']):2d}  I:{len(t[t['compound']=='INTERMEDIATE']):2d}")

Stint data ready: 486 stints across 0 races
  Ferrari     : 122 stints | S:15  M:60  H:40  I: 7
  McLaren     : 121 stints | S:19  M:55  H:39  I: 8
  Mercedes    : 122 stints | S:22  M:53  H:39  I: 8
  Red Bull    : 121 stints | S:25  M:55  H:34  I: 7


In [ ]:
# Reusable graph style
GRAPH_STYLE = dict(
    plot_bgcolor="#0f0f0f",
    paper_bgcolor="#0f0f0f",
    font=dict(color="#E8E8E8", family="Arial, sans-serif", size=14),
    height=650,
)

# Download logos and colorize them with team colors
import requests
import base64
from io import BytesIO
from PIL import Image

def load_and_colorize(url, hex_color):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGBA")
        r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)
        pixels = img.load()
        for y in range(img.height):
            for x in range(img.width):
                pr, pg, pb, pa = pixels[x, y]
                if pa > 0:
                    brightness = max(pr, pg, pb) / 255
                    pixels[x, y] = (int(r * brightness), int(g * brightness), int(b * brightness), pa)
        buf = BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/png;base64,{b64}"
    except Exception as e:
        print(f"  Error: {e}")
        return None

def load_image_b64(url):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        b64 = base64.b64encode(resp.content).decode("utf-8")
        content_type = resp.headers.get("Content-Type", "image/png")
        return f"data:{content_type};base64,{b64}"
    except Exception as e:
        print(f"  Error: {e}")
        return None

# Logo URLs (white originals from F1 CDN)
logo_urls = {
    "Ferrari": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/ferrari/2026ferrarilogowhite.webp",
    "McLaren": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/mclaren/2026mclarenlogowhite.webp",
    "Mercedes": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/mercedes/2026mercedeslogowhite.webp",
    "Red Bull": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/redbullracing/2026redbullracinglogowhite.webp",
}

# Car URLs (F1 CDN)
car_urls = {
    "Ferrari": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/ferrari/2026ferraricarright.webp",
    "McLaren": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/mclaren/2026mclarencarright.webp",
    "Mercedes": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/mercedes/2026mercedescarright.webp",
    "Red Bull": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/redbullracing/2026redbullracingcarright.webp",
}

# Colorize logos with team colors
print("Loading colored team logos...")
TEAM_LOGOS = {}
for team, url in logo_urls.items():
    img = load_and_colorize(url, TEAM_COLORS[team])
    if img:
        TEAM_LOGOS[team] = img
        print(f"  {team}: loaded + colorized")
    else:
        print(f"  {team}: FAILED")

print("\nLoading team car images...")
TEAM_CARS = {}
for team, url in car_urls.items():
    img = load_image_b64(url)
    if img:
        TEAM_CARS[team] = img
        print(f"  {team}: loaded")
    else:
        print(f"  {team}: FAILED")

print("\nDone!")

def add_team_images(fig, teams, image_dict=None, y_position=-0.15, logo_size=0.12):
    if image_dict is None:
        image_dict = TEAM_LOGOS
    for i, team in enumerate(teams):
        if team in image_dict:
            fig.add_layout_image(dict(
                source=image_dict[team],
                x=i, y=y_position,
                xref="x", yref="paper",
                sizex=0.6, sizey=logo_size,
                xanchor="center", yanchor="top",
                layer="above",
            ))

def style_axis(fig, y_title, y_range=None, y_dtick=5):
    fig.update_xaxes(
        showgrid=False, showline=False,
        tickfont=dict(size=15, color="#bbb", family="Arial, sans-serif"),
    )
    fig.update_yaxes(
        title=dict(text=y_title, font=dict(size=14, color="#777", family="Arial, sans-serif")),
        showgrid=True, gridcolor="rgba(255,255,255,0.06)", gridwidth=1,
        dtick=y_dtick, tickfont=dict(size=12, color="#555"),
        zeroline=False, showline=False,
        range=y_range,
    )

Loading colored team logos...
  Ferrari: loaded + colorized
  McLaren: loaded + colorized
  Mercedes: loaded + colorized
  Red Bull: loaded + colorized

Loading team car images...
  Ferrari: loaded
  McLaren: loaded
  Mercedes: loaded
  Red Bull: loaded

Done!


# II: Tire Compound and Degradation

Tyre management plays a major role in Formula 1 race strategy, influencing pace, pit stop timing, overtaking opportunities, and overall race consistency. Different tyre compounds provide different balances between grip and durability, meaning teams must decide whether to prioritise short-term pace or longer-lasting performance across a stint.

The following section analyses how Ferrari, McLaren, Mercedes, and Red Bull used and managed the soft, medium, and hard compounds during the 2025 season. Using stint data and lap-by-lap timing information, the graphs compare compound usage, stint lengths, and long-run pace change to identify differences in tyre strategy and degradation behaviour between the teams.

In [ ]:
# GRAPH 1: Compound Usage Profile (Dry Only)
dry = top4[top4["compound"].isin(["SOFT", "MEDIUM", "HARD"])].copy()
teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

compound_laps = dry.groupby(["team_clean", "compound"])["stint_length"].sum().reset_index(name="total_laps")
team_totals = compound_laps.groupby("team_clean")["total_laps"].sum().rename("team_total")
compound_laps = compound_laps.merge(team_totals, on="team_clean")
compound_laps["pct"] = compound_laps["total_laps"] / compound_laps["team_total"] * 100

fig = go.Figure()

for compound in ["SOFT", "MEDIUM", "HARD"]:
    vals, texts = [], []
    for team in teams_order:
        row = compound_laps[(compound_laps["team_clean"] == team) & (compound_laps["compound"] == compound)]
        if len(row) > 0:
            pct, laps = row["pct"].values[0], int(row["total_laps"].values[0])
            vals.append(pct)
            texts.append(f"<b>{pct:.0f}%</b><br><span style='font-size:11px'>{laps} laps</span>")
        else:
            vals.append(0)
            texts.append("")

    fig.add_trace(go.Bar(
        name=compound, x=teams_order, y=vals,
        text=texts, textposition="inside",
        textfont=dict(size=14, color="#111", family="Inter, sans-serif"),
        marker=dict(color=COMPOUND_COLORS[compound],
                    line=dict(color="rgba(0,0,0,0.4)", width=0.8)),
        width=0.55,
    ))

# Team accent lines
for i, team in enumerate(teams_order):
    fig.add_shape(type="rect", x0=i-0.3, x1=i+0.3, y0=-2, y1=-0.5,
                  fillcolor=TEAM_COLORS[team], line=dict(width=0), layer="above")

# Team logos
add_team_images(fig, teams_order, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

fig.update_layout(
    **GRAPH_STYLE,
    barmode="stack",
    title=dict(
        text=(f"<b>Dry Tyre Compound Usage By Team</b><br>"
              f"<span style='font-size:13px;color:#666;font-weight:normal'>"
              ),
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        font=dict(size=14, family="Inter, sans-serif"), bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=65, r=30, t=130, b=140),
)

style_axis(fig, y_title="Share of Race Laps (%)", y_range=[0, 108], y_dtick=25)

fig.show()

This stacked bar chart displays the percentage share of total dry race laps completed on each tire compound (soft, medium, and hard) by Ferrari, McLaren, Mercedes, and Red Bull during the 2025 season. The x-axis represents each team, while the y-axis shows the percentage of total dry-compound race laps completed on each tire type. Percentages were calculated by dividing the total laps completed on each compound by the team’s total dry race laps across the season. Only dry compounds were included, while wet and intermediate tires were excluded.

The graph reveals clear differences in compound usage between the four teams. Ferrari recorded the lowest soft compound usage at 10%, while Red Bull recorded the highest at 17%. Ferrari also completed the highest percentage of laps on the medium compound at 49%. Mercedes relied most heavily on the hard compound, with 43% of its total race laps completed on hards, the highest among the teams.

Ferrari’s distribution suggests a greater reliance on medium compounds and less dependence on soft tires compared to the other teams. Since medium tires provide a balance between pace and durability, this profile may indicate a preference for more stable stint lengths rather than heavily relying on shorter soft-tire runs.

In contrast, Red Bull showed the most aggressive compound distribution, recording both the highest soft usage and the lowest hard usage. Soft tires generally provide greater grip and faster short-term pace but degrade more quickly. Red Bull’s higher proportion of soft laps therefore suggests a greater willingness to prioritise short-term performance compared to the other teams.

Mercedes displayed the strongest reliance on harder compounds, with hard tires accounting for 43% of total laps. Their relatively low soft usage and high hard usage suggest a strategy profile centred around extending stints and maintaining pace over longer race distances rather than maximising short-term grip.

McLaren’s distribution sits between the other teams across all three compounds, without a strong reliance on any single tire type. This balanced usage profile suggests the team was flexible in compound selection across different race situations.

It is important to note that compound usage alone does not fully explain team strategy. Tire choices are also influenced by circuit characteristics, qualifying position, safety cars, weather conditions, and mandatory tire regulations. As a result, the graph measures how compounds were used across the season, but does not isolate whether those choices were entirely strategic or influenced by race circumstances.

In [ ]:
# GRAPH 2: Median Stint Length by Compound
dry = top4[top4["compound"].isin(["SOFT", "MEDIUM", "HARD"])].copy()
teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

fig = go.Figure()

for compound in ["SOFT", "MEDIUM", "HARD"]:
    medians, q1s, q3s = [], [], []
    for team in teams_order:
        subset = dry[(dry["team_clean"] == team) & (dry["compound"] == compound)]["stint_length"]
        med = subset.median()
        medians.append(med)
        q1s.append(med - subset.quantile(0.25))
        q3s.append(subset.quantile(0.75) - med)

    fig.add_trace(go.Bar(
        name=compound, x=teams_order, y=medians,
        error_y=dict(type="data", symmetric=False, array=q3s, arrayminus=q1s,
                     color="rgba(255,255,255,0.3)", thickness=1.5, width=5),
        marker=dict(color=COMPOUND_COLORS[compound],
                    line=dict(color="rgba(0,0,0,0.4)", width=0.8)),
        text=[f"<b>{m:.0f}</b>" for m in medians],
        textposition="outside",
        textfont=dict(size=14, color="#bbb", family="Arial, sans-serif"),
        width=0.22,
        hovertemplate=f"<b>%{{x}} — {compound}</b><br>Median: %{{y:.1f}} laps<extra></extra>",
    ))

# Team accent lines
for i, team in enumerate(teams_order):
    fig.add_shape(type="rect", x0=i-0.38, x1=i+0.38, y0=-0.8, y1=0,
                  fillcolor=TEAM_COLORS[team], line=dict(width=0), layer="above",
                  opacity=0.7)

# Team logos
add_team_images(fig, teams_order, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

fig.update_layout(
    **GRAPH_STYLE,
    barmode="group",
    title=dict(
        text=(f"<b>Median Stint Length By Dry Compound</b><br>"
              f"<span style='font-size:13px;color:#666;font-weight:normal'>"
              ),
        font=dict(size=24, color="#E8E8E8", family="Arial, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        font=dict(size=14, family="Arial, sans-serif"), bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=65, r=30, t=130, b=140),
)

style_axis(fig, y_title="Median Stint Length (laps)", y_range=[0, 42], y_dtick=5)

fig.show()

This bar chart compares the median number of laps completed per stint on the soft, medium, and hard compounds by Ferrari, McLaren, Mercedes, and Red Bull during the 2025 season. The x-axis lists the four teams, while the y-axis measures median stint length in laps. The error bars represent the interquartile range (Q1 to Q3), showing the spread of the middle 50% of stint lengths for each compound and team combination. Stint length was calculated as `(lap end - lap start) + 1` for every recorded tyre stint in the 2025 season dataset. Only dry compounds were included in the analysis, while wet and intermediate tyres were excluded. Stints shorter than 3 laps were also removed to exclude abnormal pit sequences such as late-race fastest lap stops, safety car events, and more.

Across all four teams, harder compounds generally produced longer stints than softer compounds. This is expected, as harder tyres are designed to prioritise durability over peak grip, allowing teams to extend stints and reduce pit stop frequency.

Ferrari recorded the shortest median stints on both the soft and hard compounds, with 16 laps on softs and 25 laps on hards. However, on mediums, Ferrari recorded a median of 20 laps, second only to McLaren. Combined with Graph 1, where Ferrari also showed the highest medium-compound usage, this suggests the team relied more heavily on medium tyres across the season. Ferrari’s shorter hard stints may indicate a tendency toward earlier pit windows on harder compounds rather than consistently extending tyre life across long runs.

McLaren achieved some of the longest stints overall, particularly on the hard compound with a median of 29 laps. Their medium-compound median of 21 laps was also the highest among the teams. The 12-lap increase between McLaren’s median soft and hard stints was the largest in the field, suggesting the team was able to extend stints substantially more when moving onto harder compounds. McLaren’s relatively moderate error bars also indicate fairly consistent stint lengths across races rather than highly variable strategy patterns. This controlled approach aligns with McLaren’s overall season profile, where the team frequently operated near the front of the field and often focused on maintaining stable race pace and track position rather than relying on aggressive short-stint strategies.

Mercedes showed an unusual distribution, with a median soft stint of 20 laps, slightly longer than its 19-lap median on mediums. This suggests Mercedes was sometimes able to extend soft-compound stints to lengths comparable with medium tyres. Their 29-lap hard stint median matched McLaren for the longest in the field, reinforcing the trend from Graph 1 where Mercedes also completed the highest percentage of race laps on hard compounds. The wider hard-compound error bars suggest Mercedes used a broader range of hard-tyre strategies depending on race circumstances. Overall, the data points toward a strategy that frequently prioritised longer and more flexible hard-tyre stints.

Red Bull recorded median stint lengths of 17 laps on softs, 19 on mediums, and 26 on hards. Combined with Graph 1, where Red Bull also recorded the highest soft-compound usage, this suggests the team relied more frequently on shorter soft-compound stints than its rivals. Red Bull’s wider hard-compound error bars indicate greater variation in stint lengths across races, suggesting a less standardised strategy profile compared to Ferrari or McLaren. Relative to McLaren’s more stable distributions, Red Bull’s strategy patterns appear more variable and pace-focused, with a greater willingness to trade tyre longevity for shorter-term performance advantages during races.

It is important to note that stint length alone does not directly measure tyre performance or degradation. Stint lengths can also be influenced by safety cars, weather interruptions, qualifying position, race traffic, pit stop timing, and race strategy. As a result, the graph measures how long teams typically remained on each compound, but does not isolate the specific reason why a stint was extended or shortened in any individual race.

In [ ]:
# Graph 3: Tyre Degradation on Different Compounds

import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]
compounds   = ["SOFT", "MEDIUM", "HARD"]
COMP_COLORS = {"SOFT": "#FF3333", "MEDIUM": "#FFC800", "HARD": "#CCCCCC"}

dry = top4[top4["compound"].isin(compounds)].copy()

# Build degradation data from CSV
all_curves = []
matched, skipped = 0, 0

for _, stint in dry.iterrows():
    sk = int(stint["session_key"])
    dn = int(stint["driver_number"]) if pd.notna(stint["driver_number"]) else None
    if not dn: skipped += 1; continue

    lap_s, lap_e = int(stint["lap_start"]) + 2, int(stint["lap_end"])
    if lap_e - lap_s < 2: skipped += 1; continue

    laps = of1_laps_df[
        (of1_laps_df["session_key"]   == sk) &
        (of1_laps_df["driver_number"] == dn) &
        (of1_laps_df["lap_number"]    >= lap_s) &
        (of1_laps_df["lap_number"]    <= lap_e)
    ].sort_values("lap_number").copy()
    if len(laps) < 3: skipped += 1; continue

    med  = laps["lap_duration"].median()
    laps = laps[(laps["lap_duration"] <= med * 1.05) & (laps["lap_duration"] >= med * 0.97)]
    if len(laps) < 3: skipped += 1; continue

    baseline = laps["lap_duration"].iloc[:2].mean()
    laps["tyre_age"] = range(1, len(laps) + 1)
    laps["deg"]      = laps["lap_duration"] - baseline
    laps["compound"] = stint["compound"]
    laps["team"]     = stint["team_clean"]
    all_curves.append(laps[["tyre_age", "deg", "compound", "team"]])
    matched += 1

deg_df = pd.concat(all_curves, ignore_index=True)

# Average, smooth, normalize
avg = (deg_df.groupby(["team", "compound", "tyre_age"])["deg"]
       .agg(["mean", "count"]).reset_index())
avg = avg[avg["count"] >= 2].sort_values(["team", "compound", "tyre_age"])
avg["smooth"] = (avg.groupby(["team", "compound"])["mean"]
                 .transform(lambda x: x.rolling(3, min_periods=1, center=True).mean()))

for team in teams_order:
    for compound in compounds:
        mask = (avg["team"] == team) & (avg["compound"] == compound)
        sub  = avg[mask].sort_values("tyre_age")
        if len(sub) > 0:
            avg.loc[mask, "smooth"] -= sub["smooth"].iloc[0]

# Display range
compound_team_max = {}
for compound in compounds:
    team_maxes = [
        avg[(avg["team"] == t) & (avg["compound"] == compound)]["tyre_age"].max()
        for t in teams_order
        if len(avg[(avg["team"] == t) & (avg["compound"] == compound)]) >= 3
    ]
    if team_maxes:
        compound_team_max[compound] = min(team_maxes)

DISPLAY_MAX = min(max(compound_team_max.values(), default=20), 28)
avg = avg[avg["tyre_age"] <= DISPLAY_MAX]

# Y-axis range
g_min = avg["smooth"].min()
g_max = avg["smooth"].max()
abs_m = max(abs(g_min), abs(g_max))
pad   = abs_m * 0.25
YMIN  = round(g_min - pad, 2)
YMAX  = round(g_max + pad, 2)
if YMIN > -abs_m * 0.10:
    YMIN = round(-abs_m * 0.10 - pad * 0.5, 2)
if YMAX < abs_m * 0.10:
    YMAX = round( abs_m * 0.10 + pad * 0.5, 2)

# Compute slopes
team_slopes = {}
for team in teams_order:
    team_slopes[team] = {}
    for compound in compounds:
        sub = avg[(avg["team"] == team) & (avg["compound"] == compound)].sort_values("tyre_age")
        if len(sub) >= 3:
            team_slopes[team][compound] = np.polyfit(sub["tyre_age"], sub["smooth"], 1)[0]

# Reusable chart builder
def make_deg_chart(compound):
    cc = COMP_COLORS[compound]
    fig = go.Figure()

    for ti, team in enumerate(teams_order):
        hc = TEAM_COLORS[team]
        r, g, b = int(hc[1:3], 16), int(hc[3:5], 16), int(hc[5:7], 16)
        sub = avg[(avg["team"] == team) & (avg["compound"] == compound)].sort_values("tyre_age")
        if len(sub) < 3:
            continue

        slope = team_slopes[team].get(compound, 0)
        end_y = float(sub["smooth"].iloc[-1])
        end_x = float(sub["tyre_age"].iloc[-1])

        # Fill area
        fig.add_trace(go.Scatter(
            x=pd.concat([sub["tyre_age"], sub["tyre_age"][::-1]]),
            y=pd.concat([sub["smooth"], pd.Series([0]*len(sub))]),
            fill="toself",
            fillcolor=f"rgba({r},{g},{b},0.05)",
            line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))

        # Main trend line
        fig.add_trace(go.Scatter(
            x=sub["tyre_age"],
            y=sub["smooth"],
            mode="lines",
            line=dict(color=hc, width=4.5, shape="spline", smoothing=1.2),
            name=team,
            showlegend=True,
            legendgroup=team,
            hovertemplate=(
                f"<b>{team}</b> · Lap %{{x:.0f}}<br>"
                "Delta <b>%{y:+.3f}s</b><br>"
                f"Deg rate <b>{slope:+.3f} s/lap</b>"
                "<extra></extra>"
            ),
        ))

        # Endpoint dot + slope label
        fig.add_trace(go.Scatter(
            x=[end_x], y=[end_y],
            mode="markers+text",
            marker=dict(color=hc, size=10, line=dict(color="#0f0f0f", width=2)),
            text=[f"  {slope:+.3f} s/lap"],
            textposition="middle right",
            textfont=dict(size=11, color=hc, family="Inter, sans-serif"),
            showlegend=False, hoverinfo="skip",
        ))

    fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.08)", width=1), layer="below")

    fig.add_annotation(
        text="SLOWER (DEGRADING)", x=0.005, y=0.98,
        xref="paper", yref="paper", showarrow=False,
        font=dict(size=9, color="rgba(248,113,113,0.35)", family="Inter, sans-serif"),
        xanchor="left", yanchor="top",
    )
    fig.add_annotation(
        text="FASTER (FUEL BURN-OFF)", x=0.005, y=0.02,
        xref="paper", yref="paper", showarrow=False,
        font=dict(size=9, color="rgba(74,222,128,0.35)", family="Inter, sans-serif"),
        xanchor="left", yanchor="bottom",
    )

    fig.update_layout(
        **GRAPH_STYLE,
        title=dict(
            text=f"<b>Tyre Degradation</b>  <span style='color:{cc};font-size:18px'>{compound}</span>",
            font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
            x=0.5, xanchor="center", y=0.95,
        ),
        margin=dict(l=65, r=80, t=110, b=90),
        hovermode="x unified",
        hoverlabel=dict(
            bgcolor="rgba(12,12,12,0.95)", bordercolor="#333",
            font=dict(size=12, family="Inter, sans-serif", color="#ddd"),
            namelength=0,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.01,
            xanchor="center", x=0.5,
            font=dict(size=14, family="Inter, sans-serif", color="#bbb"),
            bgcolor="rgba(0,0,0,0)",
            itemsizing="constant",
            tracegroupgap=60,
        ),
    )

    fig.update_xaxes(
        showgrid=True, gridcolor="rgba(255,255,255,0.03)",
        showline=False, zeroline=False, dtick=5,
        range=[0.5, DISPLAY_MAX + 2],
        tickfont=dict(size=13, color="#bbb", family="Inter, sans-serif"),
        title=dict(text="Laps on Tyre", font=dict(size=13, color="#777", family="Inter, sans-serif"), standoff=10),
    )
    fig.update_yaxes(
        range=[YMIN, YMAX],
        showgrid=True, gridcolor="rgba(255,255,255,0.06)", gridwidth=1,
        dtick=0.25, tickformat="+.2f", ticksuffix="s",
        tickfont=dict(size=11, color="#555", family="Inter, sans-serif"),
        zeroline=False, showline=False,
        title=dict(text="Pace Delta", font=dict(size=13, color="#777", family="Inter, sans-serif"), standoff=14),
    )

    fig.show()

## Tyre Degradation Methodology

This graph tracks how each team's lap times change over a tyre stint during the 2025 season. A stint is the set of consecutive laps a driver completes on one set of tyres between pit stops. Only dry compounds are included. Raw lap times include noise from pit stops, safety cars, traffic, and other race events. To fix this, the first 2 laps of each stint were excluded because they include time spent driving through the pit lane at reduced speed and the tyres have not yet reached their optimal operating temperature, making those laps significantly slower than normal racing pace. Stints with fewer than 3 laps were excluded. It is also important to note that other factors such as fuel load, track conditions, and other confounding variables may also influence the curves.

&nbsp;

### Pace Delta (Y-Axis)
`Pace Delta = Lap Time − Baseline`

| Variable | What it means |
|---|---|
| **Lap Time** |      Duration of a single lap (seconds)|
| **Baseline** |      The average of the first 2 lap times in the stint |
| **Pace Delta** |    How much slower or faster a lap is compared to the baseline |

&nbsp;

### Degradation Rate (Slope Label)
`y = mx + b`

| Variable | Represents | Unit |
|---|---|---|
| **y** | Pace delta | How much slower or faster a lap is compared to the baseline |
| **x** | Tyre age | Stint length |
| **m** | Slope |  Pace change per lap on the tyre |
| **b** | Intercept | Starting point of the fitted line |

In [ ]:
# GRAPH 3A: Soft tyre degradation
make_deg_chart("SOFT")

This line chart models how each team’s pace changed over the course of a soft tyre stint during the 2025 season. The x-axis represents tyre age in laps, while the y-axis shows pace delta in seconds relative to the beginning of the stint. Positive values indicate lap times became slower over time, while negative values indicate lap times became faster, usually due to fuel burn-off reducing the car’s weight throughout the race.

Ferrari recorded the highest soft-compound degradation slope at +0.057 s/lap. Their pace delta remained relatively stable during the opening phase of the stint but increased sharply after around lap 10, eventually reaching approximately +1.5 seconds by lap 23. This indicates Ferrari’s pace worsened more rapidly over long soft stints compared to the other teams. This trend also aligns with earlier graphs, where Ferrari recorded both the lowest soft-compound usage and the shortest median soft stints. Together, these results suggest Ferrari generally avoided extending soft stints because pace loss became more significant later in the run.

Mercedes recorded the second-highest degradation slope at +0.047 s/lap. Their curve rises more gradually than Ferrari’s during the middle phase of the stint before increasing more sharply near the end, finishing at approximately +1.1 seconds by lap 22. This suggests Mercedes maintained relatively stable pace early in the stint before experiencing larger late-stint pace loss.

McLaren and Red Bull recorded the lowest degradation slopes at +0.004 s/lap and +0.005 s/lap respectively. Both teams maintained stable pace profiles across their soft stints, with much smaller increases in pace delta than Ferrari or Mercedes. McLaren’s curve fluctuates within a narrow range and finishes slightly below its baseline pace, while Red Bull’s curve remains close to zero across the longest displayed soft stint range in the graph. These trends suggest both teams maintained more consistent lap times and were able to manage their tyres more effectively throughout soft-compound runs. More stable soft-tyre performance can provide greater strategic flexibility, as teams are better able to extend stints without experiencing large pace loss.

The graph overall highlights clear differences in soft-tyre stint behaviour between the teams. Ferrari and Mercedes experienced larger increases in lap time as stints progressed, while McLaren and Red Bull maintained more stable pace profiles across longer runs.

In [ ]:
# GRAPH 3B: Medium Tyre Degradation
make_deg_chart("MEDIUM")

Unlike the soft tyre graph, McLaren, Mercedes, and Ferrari recorded negative overall degradation slopes on the medium compound. This means the reduction in lap times generally from fuel burn-off generally outweighed the pace loss from tyre wear across medium stints, allowing the cars to become slightly faster on average as the stint progressed. Fuel burn-off refers to the car becoming lighter as fuel is consumed during the race. A lighter car is generally faster because there is less mass to accelerate, brake, and carry through corners. However, Red Bull was the exception, recording a small positive slope.

McLaren recorded the lowest medium-compound slope at -0.021 s/lap. Their curve remains relatively stable during the opening half of the stint before gradually decreasing later in the run, finishing at approximately -0.75 seconds by lap 28. The line also shows fewer fluctuations than the other teams, suggesting McLaren maintained very consistent medium-tyre pace throughout long runs. This aligns with earlier graphs where McLaren completed the longest median medium stints and showed balanced compound usage across the season. Together, these trends suggest the team was comfortable extending medium stints while maintaining stable pace, which would help reduce pit stop pressure and maintain their top track positions during the season.

Mercedes recorded the second-lowest slope at -0.017 s/lap. Although their curve fluctuates more than McLaren’s, the overall trend still moves downward and finishes near -1.0 seconds by lap 28, the lowest final point on the graph. This suggests Mercedes could maintain strong pace deep into long medium stints even if their lap times varied more throughout the stint duration.

Ferrari recorded a slope of -0.015 s/lap. Their pace remained relatively controlled through most of the stint before decreasing further during the final laps. Ferrari also recorded both the highest medium-compound usage in Graph 1 and one of the longer median medium stints in Graph 2. Together, these trends suggest the medium compound was likely Ferrari’s most stable and reliable race tyre during the 2025 season. Compared to their soft-tyre graph, Ferrari’s medium curve is also significantly flatter, showing much better long-run pace stability on mediums than on softs.

Red Bull was the only team to record a positive medium-compound slope at +0.007 s/lap. Their curve rises gradually through the middle phase of the stint and remains above zero for much of the run before returning closer to baseline near the end. Although the overall pace loss is still relatively small compared to Ferrari and Mercedes on soft tyres, Red Bull’s medium-tyre pace appears less stable than the other teams. This also matches earlier graphs where Red Bull relied more heavily on soft compounds and completed slightly shorter medium stints than Ferrari or McLaren. The combination of higher soft usage and weaker medium degradation performance suggests Red Bull may have preferred making pace through softer compounds rather than relying on medium-tyre runs.

Overall, the medium compound produced more stable pace trends than the soft tyre across most teams. McLaren, Mercedes, and Ferrari all recorded negative medium-compound slopes, meaning the pace gained from the car becoming lighter outweighed the pace lost from tyre wear across the stint. Red Bull, however, recorded a small positive slope, indicating a slight overall pace deterioration relative to the other teams. These results suggest the medium tyre was likely the most stable and strategically flexible compound across the 2025 season.

In [ ]:
# GRAPH 3C: Hard Tyre Degradation
make_deg_chart("HARD")

All four teams recorded negative slopes on the hard compound, meaning lap time generally decreased over the course of the stint. This indicates that the pace gained from fuel burn-off outweighed the pace lost from tyre wear during long hard-tyre runs. Compared to the soft and medium compound graphs, the hard tyre produced the most stable overall pace behaviour and the lowest degradation across the field.

Mercedes recorded the steepest negative slope at -0.033 s/lap, finishing approximately 1.24 seconds below baseline by lap 28. Their curve declines steadily throughout the stint and becomes steeper during the later laps, indicating that pace continued improving relative to the opening phase of the stint. This aligns with Graph 1 and Graph 2, where Mercedes completed the highest proportion of laps on the hard compound and also recorded one of the longest median hard stints. Together, these trends suggest Mercedes prefered managing long hard-tyre runs throughout the season.

Red Bull recorded the second-steepest slope at -0.018 s/lap, finishing around -0.6 seconds by lap 28. Compared to their medium-compound graph, where pace remained closer to baseline, the hard tyre produced more stable long-run behaviour. The curve also remains relatively smooth throughout the stint, suggesting Red Bull maintained consistent pace across extended hard-compound runs without large fluctuations in performance.

McLaren recorded a slope of -0.012 s/lap and finished at approximately -0.50 seconds by lap 28. Their curve stays within a relatively narrow range throughout most of the stint, showing gradual and controlled pace evolution rather than large swings in performance. Unlike Mercedes, McLaren does not show a sharp late-stint pace improvement, but instead maintains a more uniform trend across the run. This matches the broader pattern seen in earlier graphs, where McLaren consistently showed balanced tyre usage and stable degradation behaviour across all three compounds.

Ferrari recorded the highest negative slope at -0.009 s/lap, finishing around -0.45 seconds by lap 28. After an initial decrease during the opening laps, the curve remains comparatively flat through much of the middle section before declining slightly again near the end of the stint. Compared to the other teams, Ferrari showed the smallest overall pace improvement relative to baseline on the hard compound. This also aligns with Graph 2, where Ferrari recorded the shortest median hard stints, suggesting the team may have been less willing to extend long hard-tyre runs compared to Mercedes or McLaren.

Across all three degradation graphs, the hard compound produced the most stable long-run pace behaviour overall. Ferrari and Mercedes showed the largest change in behaviour when moving from soft tyres to harder compounds, while McLaren remained comparatively stable across all three tyre types. Red Bull showed stronger long-run stability on softs and hards than on mediums, where they were the only team to record a positive overall slope.

# III: Pit Stop Efficiency

Pit stops are a critical component of race strategy in Formula 1, where even small differences in execution and timing can determine whether a team gains or loses track position. A pit stop involves the car entering the pit lane, stopping for a tyre change performed by the crew, and then rejoining the race. The total time spent in the pit lane typically costs between 20 and 25 seconds depending on the circuit's pit lane length and speed limit, and during this time rival cars on track continue at racing speed. Beyond the physical stop itself, the strategic decision of when to pit and how many stops to make across a race distance directly affects track position, tyre condition, and overall race outcome. Teams must balance the time lost in the pit lane against the performance gained from fresh tyres, while also reacting to safety cars, weather changes, and the strategies of competitors.

This section examines pit stop performance across the four teams through three perspectives: the consistency and speed of pit stop execution, the number of stops each team chose per race, and the effect those stops had on track position in the laps immediately following.

In [ ]:
# GRAPH 4: Pit Lane Duration Distribution
import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

# Driver Lookup
drv_abbr = {}
drv_team_lookup = {}
for _, row in drivers_df.drop_duplicates(subset=["driver_number"]).iterrows():
    dn = int(row["driver_number"])
    drv_abbr[dn] = row.get("name_acronym", row.get("broadcast_name", str(dn)))
    drv_team_lookup[dn] = str(row.get("team_name", ""))

# Built Pit Data from CSV
def match_team_dn(dn):
    tn = drv_team_lookup.get(dn, "")
    for t in teams_order:
        if t.lower() in tn.lower():
            return t
    return None

pits = of1_pits_df[of1_pits_df["pit_duration"].notna() & (of1_pits_df["pit_duration"] > 0)].copy()
pits["team"] = pits["driver_number"].apply(match_team_dn)
pits["driver"] = pits["driver_number"].map(drv_abbr)
pits = pits[pits["team"].isin(teams_order)].copy()

# Stop Number per Driver per Race
pits = pits.sort_values(["session_key", "driver_number", "lap_number"])
pits["stop_num"] = pits.groupby(["session_key", "driver_number"]).cumcount() + 1

# Decide Label (stationary vs pitlane)
is_stationary = pits["pit_duration"].median() < 10
y_label = "Stationary Time (s)" if is_stationary else "Pitlane Time (s)"

# Race Name Lookup
sk_race_name = {}
for sk in pits["session_key"].unique():
    rows = of1_pits_df[of1_pits_df["session_key"] == sk]
    if "race_name" in rows.columns and len(rows) > 0:
        sk_race_name[int(sk)] = str(rows.iloc[0].get("race_name", f"R{sk}"))
    else:
        sk_race_name[int(sk)] = f"R{sk}"

pits["race_name"] = pits["session_key"].map(sk_race_name)

# Order Median from Fast to Slow
team_medians = []
for team in teams_order:
    t = pits[pits["team"] == team]["pit_duration"]
    if len(t) > 0:
        team_medians.append((team, t.median()))
team_medians.sort(key=lambda x: x[1])
ordered_teams = [t[0] for t in team_medians]


# Build Figure
fig = go.Figure()

for i, team in enumerate(ordered_teams):
    subset = pits[pits["team"] == team].copy()
    t = subset["pit_duration"].values
    hc = TEAM_COLORS[team]
    r, g, b = int(hc[1:3], 16), int(hc[3:5], 16), int(hc[5:7], 16)
    med = float(np.median(t))

    # Hover data
    race_names = subset["race_name"].fillna("").values
    drivers = subset["driver"].fillna("").values
    stop_nums = subset["stop_num"].fillna(0).astype(int).values
    lap_nums = subset["lap_number"].fillna(0).astype(int).values
    customdata = np.column_stack([race_names, drivers, stop_nums, lap_nums])

    # Jitter
    np.random.seed(42 + i)
    jitter = np.random.uniform(-0.2, 0.2, size=len(t))

    # Scatter points
    fig.add_trace(go.Scatter(
        x=[i + j for j in jitter],
        y=t,
        mode="markers",
        marker=dict(
            color=f"rgba({r},{g},{b},0.5)",
            size=7,
            line=dict(color=f"rgba({r},{g},{b},0.8)", width=0.5),
        ),
        customdata=customdata,
        showlegend=False,
        hovertemplate=(
            f"<b>{team}</b><br>"
            "<b>%{customdata[1]}</b> · %{customdata[0]}<br>"
            "Stop %{customdata[2]} · Lap %{customdata[3]}<br>"
            "<b>%{y:.2f}s</b>"
            "<extra></extra>"
        ),
    ))

    # Median line
    fig.add_shape(
        type="line", x0=i - 0.28, x1=i + 0.28, y0=med, y1=med,
        line=dict(color=hc, width=2.5),
    )

    # Median label
    fig.add_annotation(
        x=i, y=med,
        text=f"<b>{med:.2f}s</b>",
        showarrow=True, arrowhead=0, arrowwidth=1,
        arrowcolor="rgba(255,255,255,0.15)",
        ax=0, ay=-28,
        font=dict(size=13, color="#eee", family="Inter, sans-serif"),
        bgcolor="rgba(15,15,15,0.8)", borderpad=3,
    )

# Team Logos
add_team_images(fig, ordered_teams, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

# Layotu
fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text=f"<b>Pit Lane Time Distribution</b>",
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    margin=dict(l=65, r=30, t=100, b=140),
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(len(ordered_teams))),
        ticktext=ordered_teams,
        showgrid=False, zeroline=False,
    ),
    yaxis=dict(zeroline=False),
)

style_axis(fig, y_title=y_label, y_dtick=5)
fig.update_xaxes(showline=False, zeroline=False)
fig.update_yaxes(showline=False, zeroline=False)

fig.show()

This dot plot displays the distribution of individual pit lane durations for Ferrari, McLaren, Mercedes, and Red Bull across the 2025 season, with each dot representing a single pit stop event. Pit lane duration was measured using the `OpenF1 pit_duration` variable, which records the total elapsed time from pit entry to pit exit for each pit stop event. The y-axis measures total pit lane time in seconds, including the full transit from pit entry to pit exit rather than only the stationary tyre change. The graph measures the total time lost while entering, travelling through, and exiting the pit lane, not just pit crew performance. Teams are ordered from left to right by median pit lane duration, with the horizontal line across each group representing the team median.

The overall spread between the four teams is very small, with median pit lane durations ranging from 22.83 seconds to 23.09 seconds between the teams. The difference between the fastest and slowest median is only 0.26 seconds, suggesting all four teams operated at a similarly competitive level throughout the season.

Ferrari and Red Bull recorded the joint-fastest median pit lane duration at 22.83 seconds. Ferrari’s data points are clustered more tightly around the median, particularly between approximately 21 and 24 seconds, suggesting relatively consistent pit lane performance across most races. Red Bull’s distribution is noticeably wider, with more stops extending above 28 seconds and several stops above 30 seconds. This larger spread suggests Red Bull experienced greater variation in total pit lane time.

McLaren recorded a median pit lane duration of 23.04 seconds, only 0.21 seconds slower than Ferrari and Red Bull. Most McLaren pit lane durations remain concentrated close to the median, with fewer very high-duration outliers compared to Ferrari and Red Bull. Although McLaren’s typical pit lane duration was slightly slower overall, the narrower spread suggests stable performance across different pit lane situations.

Mercedes recorded the slowest median pit lane duration at 23.09 seconds, although the difference relative to the other teams remains extremely small. Their distribution shows fewer very high-duration stops above 30 seconds compared to Ferrari, McLaren, and Red Bull. This suggests Mercedes generally avoided the largest pit lane delays even if their median performance was marginally slower overall. Most Mercedes pit lane durations also remain grouped between approximately 20 and 25 seconds, indicating a controlled and stable performance across the season.

Several outlier points above 30 seconds are visible across all four teams. Since the graph measures full pit lane duration rather than stationary tyre change time, these unusually long stops were likely influenced by factors such as pit lane traffic, double stacking, front wing changes, safety car conditions, long pit lane layouts, or delayed releases rather than only slow tyre changes. Because all four teams display a small number of these outliers, the graph suggests unusually long pit lane losses were occasional race events rather than a persistent weakness for any individual team.

In [ ]:
# GRAPH 5: Number of Pitstops Strategy per Race
teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

# Map Drivers to Teams
def match_team(cn):
    cn = str(cn).lower()
    if "ferrari" in cn: return "Ferrari"
    if "mclaren" in cn: return "McLaren"
    if "mercedes" in cn: return "Mercedes"
    if "red bull" in cn: return "Red Bull"
    return None

driver_team = results_df[["driverId", "constructorName"]].drop_duplicates().copy()
driver_team["team"] = driver_team["constructorName"].apply(match_team)
dt_map = dict(zip(driver_team.dropna(subset=["team"])["driverId"],
                   driver_team.dropna(subset=["team"])["team"]))

# Count Stops per Driver per Race, Then Group by Team
pits = pitstops_df.copy()
pits["team"] = pits["driverId"].map(dt_map)
pits = pits[pits["team"].isin(teams_order)].copy()

stops_per_entry = (pits.groupby(["round", "driverId", "team"])["stop"]
                   .max().reset_index().rename(columns={"stop": "num_stops"}))

# Cap at 3+
stops_per_entry["strategy"] = stops_per_entry["num_stops"].clip(upper=3)
stops_per_entry["strategy_label"] = stops_per_entry["strategy"].map(
    {1: "1-STOP", 2: "2-STOP", 3: "3+ STOP"}
)

# Build Percentage Data for Stacked Bar
strategy_order = ["1-STOP", "2-STOP", "3+ STOP"]
STRATEGY_COLORS = {
    "1-STOP":   "rgb(80, 200, 120)",
    "2-STOP":   "rgb(255, 180, 50)",
    "3+ STOP":  "rgb(255, 65, 65)",
}

fig = go.Figure()

for strat in strategy_order:
    vals, texts = [], []
    for team in teams_order:
        t = stops_per_entry[stops_per_entry["team"] == team]
        total = len(t)
        count = len(t[t["strategy_label"] == strat])
        pct = count / total * 100 if total > 0 else 0
        vals.append(pct)
        texts.append(f"<b>{pct:.0f}%</b><br><span style='font-size:11px'>{count} races</span>")

    fig.add_trace(go.Bar(
        name=strat, x=teams_order, y=vals,
        text=texts, textposition="inside",
        textfont=dict(size=14, color="#111", family="Inter, sans-serif"),
        marker=dict(color=STRATEGY_COLORS[strat],
                    line=dict(color="rgba(0,0,0,0.4)", width=0.8)),
        width=0.55,
        hovertemplate=f"<b>%{{x}} — {strat}</b><br>%{{y:.1f}}% of races<extra></extra>",
    ))

# Team Accent Bars
for i, team in enumerate(teams_order):
    fig.add_shape(type="rect", x0=i-0.3, x1=i+0.3, y0=-2, y1=-0.5,
                  fillcolor=TEAM_COLORS[team], line=dict(width=0), layer="above")

# Team Logos
add_team_images(fig, teams_order, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

# Layout
fig.update_layout(
    **GRAPH_STYLE,
    barmode="stack",
    title=dict(
        text=(
            f"<b>Pit Stop Strategy Distribution</b><br>"
            f"<span style='font-size:13px;color:#666;font-weight:normal'>"
        ),
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        font=dict(size=14, family="Inter, sans-serif"), bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=65, r=30, t=130, b=140),
)

style_axis(fig, y_title="Share of Races (%)", y_range=[0, 108], y_dtick=25)

fig.show()

This stacked bar chart shows the percentage of driver-race entries in which Ferrari, McLaren, Mercedes, and Red Bull used a 1-stop, 2-stop, or 3+ stop strategy during the 2025 season. The x-axis lists the four teams, while the y-axis represents the percentage share of races. Race counts are displayed inside each segment.

Across all four teams, the 1-stop strategy was the most common approach. However, the balance between 1-stop, 2-stop, and 3+ stop races reveals noticeable differences in how each team managed tyre life and race strategy throughout the season.

Red Bull used a 1-stop strategy most frequently at 52% of races, the highest among the four teams, while also recording the lowest percentage of 2-stop races at 30%. Earlier tyre degradation graphs showed Red Bull maintained relatively stable long-run pace, particularly on soft tyres, which may have helped support longer stints and fewer pit stops. Since pit stops cost significant time due to pit lane transit, reducing the number of stops can help maintain track position during races.

McLaren showed the most balanced strategy distribution, with 47% 1-stop races and 43% 2-stop races. Their low percentage of 3+ stop races at 11%, the lowest among all teams, suggests McLaren generally avoided highly aggressive or heavily reactive strategies. Earlier graphs also showed stable tyre degradation across all compounds, which may have allowed the team to adapt strategies depending on race conditions rather than relying heavily on one specific approach. During the 2025 season, McLaren frequently qualified near the front, so maintaining track position may have reduced the need for aggressive multi-stop recovery strategies.

Mercedes used a 1-stop strategy in 49% of races, second only to Red Bull, but also recorded the highest percentage of 3+ stop races at 19%. Earlier graphs showed Mercedes performed strongly on long hard-tyre stints, which helps explain their high proportion of 1-stop races. However, the larger share of 3+ stop strategies also suggests Mercedes occasionally shifted toward more aggressive or flexible race approaches depending on race circumstances.

Ferrari recorded the most balanced strategy split overall, with 43% of races using a 1-stop strategy and 41% using a 2-stop strategy. Unlike Red Bull, which relied more heavily on long 1-stop races, Ferrari switched between both approaches at a similar rate throughout the season. Earlier graphs also showed Ferrari experienced higher soft-tyre pace loss and shorter average stint lengths,. This may have made long single-stop races more difficult to maintain consistently, leading Ferrari to rely more often on flexible 2-stop strategies depending on tyre wear and race conditions.

In [ ]:
# GRAPH 6 — Position Change After Pitstops (Real Lap Positions)
import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]
LAPS_AFTER = 10

drv_team = {}
for _, row in drivers_df.drop_duplicates(subset=["driver_number"]).iterrows():
    dn = int(row["driver_number"])
    tn = str(row.get("team_name", ""))
    for t in teams_order:
        if t.lower() in tn.lower():
            drv_team[dn] = t
            break

laps = of1_laps_df.copy()
laps = laps.sort_values(["session_key", "driver_number", "lap_number"])
laps["cum_time"] = laps.groupby(["session_key", "driver_number"])["lap_duration"].cumsum()
laps["position"] = (laps.groupby(["session_key", "lap_number"])["cum_time"]
                    .rank(method="min").astype(int))

pos_lookup = {}
for _, row in laps.iterrows():
    pos_lookup[(int(row["session_key"]), int(row["driver_number"]), int(row["lap_number"]))] = int(row["position"])

sk_names = laps_enriched_df[["session_key", "circuit_short_name"]].drop_duplicates()
sk_race_name = dict(zip(sk_names["session_key"].astype(int), sk_names["circuit_short_name"]))
for _, row in of1_pits_df.iterrows():
    sk = int(row["session_key"])
    if sk not in sk_race_name:
        sk_race_name[sk] = str(row.get("race_name", f"R{sk}"))

pits = of1_pits_df.copy()
pits["team"] = pits["driver_number"].map(drv_team)
pits = pits[pits["team"].isin(teams_order)].copy()
pits["lap_number"] = pd.to_numeric(pits["lap_number"], errors="coerce")
pits = pits[pits["lap_number"].notna()].copy()
pits["lap_number"] = pits["lap_number"].astype(int)
pits["session_key"] = pits["session_key"].astype(int)
pits["driver_number"] = pits["driver_number"].astype(int)

deltas = []
for _, pit in pits.iterrows():
    sk, dn, pit_lap = pit["session_key"], pit["driver_number"], pit["lap_number"]
    p_before = pos_lookup.get((sk, dn, pit_lap))
    p_after = pos_lookup.get((sk, dn, pit_lap + LAPS_AFTER))
    if p_before is None or p_after is None:
        continue
    deltas.append({
        "team": pit["team"], "session_key": sk, "pit_lap": pit_lap,
        "pos_before": p_before, "pos_after": p_after,
        "delta": p_before - p_after,
        "race": sk_race_name.get(sk, f"R{sk}"),
    })

delta_df = pd.DataFrame(deltas)

team_stats = []
for team in teams_order:
    t = delta_df[delta_df["team"] == team]
    if len(t) == 0:
        continue
    team_stats.append({
        "team": team, "avg_delta": t["delta"].mean(), "count": len(t),
        "gained": (t["delta"] > 0).sum(),
        "lost": (t["delta"] < 0).sum(),
        "neutral": (t["delta"] == 0).sum(),
    })

team_stats.sort(key=lambda x: x["avg_delta"], reverse=True)
ordered_teams = [s["team"] for s in team_stats]

# ── Chart ──
fig = go.Figure()

for stat in team_stats:
    team = stat["team"]
    hc = TEAM_COLORS[team]
    delta = stat["avg_delta"]
    r, g, b = int(hc[1:3], 16), int(hc[3:5], 16), int(hc[5:7], 16)

    t = delta_df[delta_df["team"] == team].copy()
    gained = t[t["delta"] > 0].sort_values("delta", ascending=False)
    lost = t[t["delta"] < 0].sort_values("delta")

    hover_lines = [
        f"<b>{team}</b>",
        f"Avg position change: <b>{delta:+.2f}</b>",
        f"{stat['count']} pit stops", ""
    ]
    if len(gained) > 0:
        hover_lines.append(f"<span style='color:#4ade80'>▲ Gained ({len(gained)}):</span>")
        for _, row in gained.head(8).iterrows():
            hover_lines.append(
                f"  {row['race']} L{row['pit_lap']}: "
                f"P{row['pos_before']}→P{row['pos_after']} ({row['delta']:+d})")
        if len(gained) > 8:
            hover_lines.append(f"  ...and {len(gained)-8} more")
    if len(lost) > 0:
        hover_lines.append(f"<span style='color:#f87171'>▼ Lost ({len(lost)}):</span>")
        for _, row in lost.head(5).iterrows():
            hover_lines.append(
                f"  {row['race']} L{row['pit_lap']}: "
                f"P{row['pos_before']}→P{row['pos_after']} ({row['delta']:+d})")
        if len(lost) > 5:
            hover_lines.append(f"  ...and {len(lost)-5} more")

    fig.add_trace(go.Bar(
        x=[team], y=[delta],
        marker=dict(color=f"rgba({r},{g},{b},0.25)",
                    line=dict(color=hc, width=2)),
        width=0.5, showlegend=False,
        hovertemplate="<br>".join(hover_lines) + "<extra></extra>",
    ))

    fig.add_annotation(x=team, y=delta,
        text=f"<b>{delta:+.2f}</b>",
        showarrow=False,
        font=dict(size=18, color="#eee", family="Inter, sans-serif"),
        yanchor="bottom" if delta >= 0 else "top",
        yshift=8 if delta >= 0 else -8)

    fig.add_annotation(x=team, y=delta,
        text=(f"<span style='color:#4ade80'>{stat['gained']} gained</span>"
              f"<span style='color:#333'>  ·  </span>"
              f"<span style='color:#f87171'>{stat['lost']} lost</span>"
              f"<span style='color:#333'>  ·  </span>"
              f"<span style='color:#666'>{stat['neutral']} same</span>"),
        showarrow=False,
        font=dict(size=10, family="Inter, sans-serif"),
        yanchor="bottom" if delta >= 0 else "top",
        yshift=35 if delta >= 0 else -35)

fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.1)", width=1))

# Team logos + names below
add_team_images(fig, ordered_teams, TEAM_LOGOS, y_position=-0.08, logo_size=0.14)

for i, team in enumerate(ordered_teams):
    fig.add_annotation(
        x=i, y=-0.22, xref="x", yref="paper",
        text=f"<b>{team}</b>",
        font=dict(color=TEAM_COLORS[team], size=13, family="Inter, sans-serif"),
        showarrow=False, xanchor="center", yanchor="top",
    )

fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text=f"<b>Position Change {LAPS_AFTER} Laps After Pitstop</b>",
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    margin=dict(l=65, r=30, t=100, b=160),
    xaxis=dict(
        categoryorder="array",
        categoryarray=ordered_teams,
        showticklabels=False,
    ),
)

style_axis(fig, y_title="Avg Positions Gained / Lost", y_dtick=0.5)
fig.show()

This bar chart shows the average number of race positions each team gained or lost ten laps after completing a pit stop during the 2025 season. The y-axis represents the average position change relative to the driver’s position on the pit stop lap. Positive values indicate a net gain in positions after the stop, while negative values indicate positions were lost over the following ten laps. The annotations below each bar show how many pit stops resulted in positions gained, lost, or unchanged. Driver positions were determined by ranking cumulative race time at the end of each lap, allowing positions to be measured directly by lap number rather than estimated using timestamps. For each pit stop, the driver’s race position on the pit lap was compared to their position ten laps later using: `Position Change = Position Before Pitstop − Position 10 Laps Later`

All four teams recorded negative average position changes over the ten-lap window, meaning drivers generally lost positions after pit stops on average. This likely reflects how pit stops naturally drop drivers into traffic or behind rivals who have not yet stopped. Over a longer ten-lap period, factors such as tyre wear differences, traffic management, safety cars, and alternate pit strategies begin to influence race position more heavily than the immediate pace advantage of fresh tyres.

McLaren recorded the smallest average loss at -0.45 positions per pit stop. Out of 76 analysed pit stops, McLaren gained positions in 16 cases, lost positions in 28, and remained unchanged in 32. Their relatively high number of unchanged outcomes suggests McLaren frequently maintained stable track position after stops rather than experiencing large swings in race order. Earlier strategy graphs also showed McLaren relied less on aggressive multi-stop strategies, which supports the idea that their pit timing was generally focused on maintaining race control and avoiding unnecessary position risk.

Ferrari recorded an average change of -0.54 positions per stop. Although Ferrari lost positions more often than they gained, their overall result remained relatively close to McLaren’s. The team recorded 14 gains, 32 losses, and 33 unchanged outcomes, showing that many Ferrari pit stops produced little overall change in race position. Earlier tyre degradation graphs also showed Ferrari experienced stronger soft-tyre degradation than the other teams, which may have made maintaining position over longer post-stop runs more difficult.

Mercedes recorded an average loss of -0.70 positions across the analysed stops. Despite earlier graphs showing strong long-run tyre performance, Mercedes lost positions in 35 cases while gaining positions in only 17. One possible explanation is that Mercedes often attempted longer stints or alternate strategies, causing drivers to rejoin further back in traffic before recovering pace later in the race. Their higher proportion of 3+ stop strategies in Graph 5 also supports the idea that Mercedes used more aggressive or variable race strategies throughout the season.

Red Bull recorded the largest average loss at -0.71 positions per stop. Although they gained positions in 18 cases, they also lost positions in 28 and remained unchanged in 26. Earlier graphs showed Red Bull frequently used long stints and one-stop strategies, particularly on softer compounds. Earlier graphs showed Red Bull frequently used long stints and one-stop strategies, particularly on softer compounds. Because longer stints often involve staying out while rivals pit earlier for fresher tyres, Red Bull drivers may have temporarily lost track position during the pit cycle before recovering later in the race. This likely contributed to the larger average position loss over the ten-lap window.

Overall, the graph suggests that gaining positions immediately after pit stops was uncommon over a longer ten-lap period. While fresh tyres can initially improve pace, race position after pit stops is also heavily influenced by traffic, alternate strategies, and the timing of rival pit stops. McLaren showed the most stable overall position retention after stops, while Mercedes and Red Bull experienced larger average losses despite strong long-run pace in earlier tyre degradation graphs.

# IV: Race Position Performance

Qualifying position and track position are important in Formula 1 because overtaking can often be difficult, especially on circuits with limited passing opportunities. Teams aim not only to achieve strong starting positions in qualifying, but also to convert those positions into strong race finishes through race pace, tyre strategy, pit stop timing, and consistency across a race distance. Maintaining track position can reduce time lost in traffic and give teams greater strategic control during races, while poor race execution can cause drivers to lose positions even after strong qualifying performances.

This section analyses how effectively each team converted qualifying performance into race results during the 2025 season. The following graphs compare starting grid positions with finishing outcomes, position changes after pit stops, and how consistently teams maintained or gained track position throughout races.

In [ ]:
# GRAPH 7: Starting Grid vs Race Finish Correlation
import numpy as np
from scipy import stats

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

def match_team(cn):
    cn = str(cn).lower()
    for t in teams_order:
        if t.lower() in cn: return t
    return None

res = results_df.copy()
res["team"] = res["constructorName"].apply(match_team)
res = res[res["team"].isin(teams_order)].copy()
res["position"] = pd.to_numeric(res["position"], errors="coerce")
res = res[res["position"].notna()].copy()
res["grid"] = res["grid"].astype(int)
res["position"] = res["position"].astype(int)
res = res[res["status"].str.contains(r"Finished|\+", na=False)].copy()

race_names = schedule_df.set_index("round")["raceName"].to_dict() if "raceName" in schedule_df.columns else {}

# Regression stats
reg_stats = {}
for team in teams_order:
    t = res[res["team"] == team]
    if len(t) > 2:
        slope, intercept, r, p, se = stats.linregress(t["grid"], t["position"])
        reg_stats[team] = {"slope": slope, "intercept": intercept, "r2": r**2, "n": len(t)}

team_order = sorted(reg_stats, key=lambda t: reg_stats[t]["r2"], reverse=True)

fig = go.Figure()

# Zone shading
fig.add_shape(type="path",
    path="M 0.5 0.5 L 20.5 20.5 L 20.5 0.5 Z",
    fillcolor="rgba(39,244,210,0.05)", line=dict(width=0), layer="below")
fig.add_shape(type="path",
    path="M 0.5 0.5 L 20.5 20.5 L 0.5 20.5 Z",
    fillcolor="rgba(232,0,45,0.05)", line=dict(width=0), layer="below")

# Zone labels
fig.add_annotation(x=19.5, y=1.5, text="▲ GAINED POSITIONS",
    font=dict(size=11, color="rgba(39,244,210,0.25)", family="Arial"),
    showarrow=False, xanchor="right", yanchor="top", textangle=0)
fig.add_annotation(x=1.5, y=19.5, text="▼ LOST POSITIONS",
    font=dict(size=11, color="rgba(232,0,45,0.25)", family="Arial"),
    showarrow=False, xanchor="left", yanchor="bottom", textangle=0)

# Reference line: finish = start
fig.add_trace(go.Scatter(
    x=[0.5, 20.5], y=[0.5, 20.5],
    mode="lines",
    line=dict(color="rgba(255,255,255,0.25)", width=2, dash="dash"),
    showlegend=False, hoverinfo="skip",
))
fig.add_annotation(
    x=20.8, y=20.8,
    text="<b>FINISH = START</b>",
    font=dict(size=10, color="rgba(255,255,255,0.35)", family="Arial"),
    showarrow=False, xanchor="right", yanchor="top",
)

np.random.seed(42)

for idx, team in enumerate(team_order):
    t = res[res["team"] == team]
    color = TEAM_COLORS[team]
    s = reg_stats[team]
    slope, intercept, r2, n = s["slope"], s["intercept"], s["r2"], s["n"]

    r_c, g_c, b_c = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)

    jitter_x = np.random.uniform(-0.25, 0.25, len(t))
    jitter_y = np.random.uniform(-0.25, 0.25, len(t))

    hover_text = []
    for _, row in t.iterrows():
        driver = (str(row.get("givenName", "")) + " " + str(row.get("familyName", ""))).strip()
        if not driver:
            driver = str(row.get("driverId", "Unknown"))
        race = race_names.get(int(row["round"]), f"Round {int(row['round'])}")
        delta = int(row["grid"] - row["position"])
        sign = "+" if delta > 0 else ""
        hover_text.append(
            f"<b>{driver}</b><br>"
            f"{race}<br>"
            f"Started: P{int(row['grid'])}  →  Finished: P{int(row['position'])}<br>"
            f"Delta: {sign}{delta}"
        )

    # Scatter dots
    fig.add_trace(go.Scatter(
        x=t["grid"].values + jitter_x,
        y=t["position"].values + jitter_y,
        mode="markers",
        name=team,
        marker=dict(
            color=f"rgba({r_c},{g_c},{b_c},0.45)",
            size=7,
            line=dict(color=f"rgba({r_c},{g_c},{b_c},0.7)", width=0.8),
        ),
        hovertemplate="%{customdata}<extra></extra>",
        customdata=hover_text,
        showlegend=False,
    ))

    # Regression line
    x_line = np.linspace(t["grid"].min(), t["grid"].max(), 50)
    y_line = slope * x_line + intercept

    reg_hover = [
        f"<b>{team}</b><br>"
        f"R² = {r2:.3f}<br>"
        f"Slope = {slope:.2f}<br>"
        f"Grid P{int(round(xv))} → Expected P{yv:.1f}<extra></extra>"
        for xv, yv in zip(x_line, y_line)
    ]

    # Glow layer
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode="lines",
        line=dict(color=f"rgba({r_c},{g_c},{b_c},0.12)", width=10),
        showlegend=False, hoverinfo="skip",
    ))
    # Main regression line (hoverable)
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode="lines",
        line=dict(color=color, width=2.5),
        showlegend=False,
        hovertemplate=reg_hover,
    ))

# Legend
legend_x = 0.99
legend_top = 0.97
row_height = 0.07

for idx, team in enumerate(team_order):
    s = reg_stats[team]
    color = TEAM_COLORS[team]
    y_pos = legend_top - (idx * row_height)

    if team in TEAM_LOGOS:
        fig.add_layout_image(dict(
            source=TEAM_LOGOS[team],
            x=legend_x - 0.27, y=y_pos,
            xref="paper", yref="paper",
            sizex=0.03, sizey=0.03,
            xanchor="left", yanchor="middle",
            layer="above",
        ))

    fig.add_annotation(
        x=legend_x - 0.23, y=y_pos,
        xref="paper", yref="paper",
        text=f"<b>{team}</b>",
        font=dict(color=color, size=11),
        showarrow=False, xanchor="left", yanchor="middle",
    )

    fig.add_annotation(
        x=legend_x - 0.23, y=y_pos - 0.022,
        xref="paper", yref="paper",
        text=f"R²={s['r2']:.2f}   slope={s['slope']:.2f}   n={s['n']}",
        font=dict(color="rgba(255,255,255,0.4)", size=9),
        showarrow=False, xanchor="left", yanchor="middle",
    )

fig.add_shape(
    type="rect",
    x0=legend_x - 0.29, y0=legend_top + 0.03,
    x1=legend_x + 0.01, y1=legend_top - (len(team_order) * row_height) + 0.03,
    xref="paper", yref="paper",
    fillcolor="rgba(10,10,10,0.85)",
    line=dict(color="rgba(255,255,255,0.06)", width=1),
    layer="below",
)

fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text="<b>Starting Grid Position vs. Race Finish Position</b>",
        font=dict(size=22, color="#E8E8E8"),
        x=0.5, xanchor="center", y=0.96,
    ),
    xaxis=dict(
        title=dict(text="Grid Position", font=dict(size=13, color="#666")),
        zeroline=False, dtick=2,
        range=[0, 21], autorange=False,
        gridcolor="rgba(255,255,255,0.03)", showgrid=True,
        tickfont=dict(size=11, color="#555"),
    ),
    yaxis=dict(
        title=dict(text="Finish Position", font=dict(size=13, color="#666")),
        zeroline=False, dtick=2,
        range=[0, 21], autorange=False,
        gridcolor="rgba(255,255,255,0.03)", showgrid=True,
        tickfont=dict(size=11, color="#555"),
    ),
    showlegend=False,
    margin=dict(t=70, b=50, l=55, r=30),
    hoverlabel=dict(
        bgcolor="rgba(15,15,15,0.92)",
        bordercolor="rgba(255,255,255,0.15)",
        font=dict(size=12, color="#E8E8E8", family="Arial"),
    ),
)

fig.show()

This scatter plot compares each driver’s starting grid position with their final race finishing position during the 2025 season. The x-axis represents grid position at the start of the race, while the y-axis represents finishing position. Each point represents a single race result for Ferrari, McLaren, Mercedes, or Red Bull drivers who finished the race. Lower finishing positions indicate better race outcomes, so points below the diagonal reference line represent races where positions were gained, while points above the line represent races where positions were lost.

Only classified race finishes were included in the analysis. Drivers who retired or failed to finish were excluded using the official race status field. Linear regression was then applied separately to each team using: `Finish Position = (Slope * Grid Position) + Intercept`. The regression line models the relationship between starting position and finishing position across the season. The graph also calculates the coefficient of determination (r²), which measures how closely finishing positions followed starting positions for each team. A higher r² value indicates that race results were more strongly determined by qualifying position, while a lower R² value suggests greater variation between starting and finishing outcomes.

Red Bull recorded the highest r² value at 0.56, meaning their finishing positions were most strongly correlated with where they started on the grid. Their regression line also has the steepest slope at 0.55. This suggests Red Bull drivers generally finished close to their starting positions, with less overall movement through the field compared to the other teams. The scatter points are distributed relatively tightly around the regression line, especially for front-running grid positions. Earlier strategy graphs also showed Red Bull frequently relied on longer stints and one-stop strategies, which often prioritise maintaining track position rather than creating large position swings during races.

Ferrari recorded an R² value of 0.30 with a slope of 0.32. Compared to Red Bull, Ferrari’s scatter points are more widely spread vertically, meaning drivers experienced larger changes between grid and finishing positions. Several Ferrari results also appear well below the reference line, showing races where significant positions were gained. However, Ferrari also displays some of the largest negative outcomes above the line, indicating greater inconsistency across race weekends.

Mercedes recorded an r² value of 0.29 and a slope of 0.50. Their regression slope remains relatively steep, suggesting starting position still had a strong influence on race finish, but the lower r² value indicates greater variability in race outcomes compared to Red Bull. Mercedes points are spread across both the gained-position and lost-position regions of the graph, particularly from midfield starting positions. This suggests Mercedes experienced more unexpected race outcomes. Earlier tyre degradation graphs also showed Mercedes performed strongly on long hard-tyre stints, which may have contributed to larger recovery drives from weaker grid positions in some races.

McLaren recorded the lowest r² value at 0.27 and the shallowest slope at 0.46. This suggests McLaren’s finishing positions were the least dependent on starting position among the four teams. Their regression line sits lower than the others across much of the grid range, meaning McLaren generally converted starting positions into stronger finishing results on average. Many McLaren points also fall below the finish-equals-start line, particularly from midfield grid positions, indicating consistent position gains during races. Earlier graphs support this interpretation, as McLaren showed stable tyre degradation across all compounds, and controlled pit lane performance throughout the season.

The dashed diagonal reference line represents races where drivers finished exactly where they started. Points below the line indicate positions gained during the race, while points above the line indicate positions lost. Across all four teams, the graph shows that qualifying position remained an important factor in final race result. It is also important to note that other confounding variables other than starting grid position like race pace, tyre management, pit strategy, and reliability significantly influenced finishing outcomes throughout the season.

In [ ]:
# GRAPH 8: Strip Plot of Front Grid Position Conversion (P1–P5 Starts)
import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

# Map drivers to teams
def match_team(cn):
    cn = str(cn).lower()
    if "ferrari" in cn: return "Ferrari"
    if "mclaren" in cn: return "McLaren"
    if "mercedes" in cn: return "Mercedes"
    if "red bull" in cn: return "Red Bull"
    return None

res = results_df.copy()
res["team"] = res["constructorName"].apply(match_team)
res = res[res["team"].isin(teams_order)].copy()

# Filter: only finished races (exclude DNFs)
res = res[res["status"].str.contains(r"Finished|\+", na=False)].copy()

# Filter: started P1–P5 only
res = res[res["grid"].between(1, 5)].copy()

# Position delta
res["delta"] = res["grid"] - res["position"]

# Race name lookup
race_names = schedule_df.set_index("round")["raceName"].to_dict() if "raceName" in schedule_df.columns else {}

# Order teams by average delta
team_avg = res.groupby("team")["delta"].mean().sort_values(ascending=False)
ordered_teams = [t for t in team_avg.index if t in teams_order]

# Build Figure
fig = go.Figure()

np.random.seed(42)

# Zero reference line
fig.add_shape(
    type="line",
    x0=-0.5, x1=len(ordered_teams) - 0.5,
    y0=0, y1=0,
    line=dict(color="rgba(255,255,255,0.3)", width=2, dash="dash"),
)

# Zone labels
fig.add_annotation(
    x=len(ordered_teams) - 0.5, y=2,
    text="<b>GAINED ▲</b>",
    font=dict(size=11, color="rgba(80,200,120,0.5)"),
    showarrow=False, xanchor="right",
)
fig.add_annotation(
    x=len(ordered_teams) - 0.5, y=-3,
    text="<b>LOST ▼</b>",
    font=dict(size=11, color="rgba(255,80,80,0.5)"),
    showarrow=False, xanchor="right",
)

# Strip dots
for i, team in enumerate(ordered_teams):
    t = res[res["team"] == team]
    if len(t) == 0:
        continue

    jitter = np.random.uniform(-0.2, 0.2, size=len(t))

    hover_text = []
    for _, row in t.iterrows():
        delta = row["delta"]
        sign = "+" if delta > 0 else ""
        driver = row.get("givenName", "") + " " + row.get("familyName", "")
        driver = driver.strip() if driver.strip() else row.get("driverId", "Unknown")
        race = race_names.get(int(row["round"]), f"Round {int(row['round'])}")
        hover_text.append(
            f"<b>{driver}</b><br>"
            f"{race}<br>"
            f"Grid: P{int(row['grid'])} → Finish: P{int(row['position'])}<br>"
            f"Delta: {sign}{int(delta)}"
        )

    fig.add_trace(go.Scatter(
        x=[i] * len(t) + jitter,
        y=t["delta"].values,
        mode="markers",
        name=team,
        marker=dict(
            color=TEAM_COLORS[team],
            size=10,
            opacity=0.75,
            line=dict(color="rgba(0,0,0,0.4)", width=1),
        ),
        hovertemplate="%{customdata}<extra></extra>",
        customdata=hover_text,
    ))

    # Average marker (diamond)
    avg_val = t["delta"].mean()
    fig.add_trace(go.Scatter(
        x=[i], y=[avg_val],
        mode="markers",
        marker=dict(
            symbol="diamond",
            color="white",
            size=14,
            line=dict(color=TEAM_COLORS[team], width=2.5),
        ),
        showlegend=False,
        hovertemplate=f"<b>{team} Average</b><br>{avg_val:+.2f} positions<extra></extra>",
    ))

# Layout
fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text="<b>Front Grid Position Conversion (P1–P5 Starts)</b>",
        font=dict(size=26, color="#E8E8E8"),
        x=0.5, xanchor="center",
    ),
    legend=dict(
        orientation="h", x=0.5, xanchor="center", y=-0.15,
        font=dict(size=13, color="#bbb"),
        bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=70, r=40, t=80, b=120),
)

fig.update_xaxes(
    tickvals=list(range(len(ordered_teams))),
    ticktext=["" for _ in ordered_teams],
    showgrid=False, showline=False,
    zeroline=False,
    range=[-0.5, len(ordered_teams) - 0.5],
)

style_axis(fig, y_title="Positions Gained / Lost", y_range=None, y_dtick=2)

add_team_images(fig, ordered_teams, TEAM_LOGOS, y_position=0.03, logo_size=0.12)

fig.show()

This strip plot analyses how effectively each team converted strong qualifying positions into race results during the 2025 season. Only races where drivers started between P1 and P5 and officially finished the race were included. Each dot represents a single race result, while the y-axis shows the number of positions gained or lost relative to starting position using: `Position Change = Grid Position − Finish Position`. Positive values indicate positions gained during the race, while negative values indicate positions lost. The white diamond marker represents each team’s average position change across all front-grid starts.

McLaren recorded the strongest average front-grid conversion overall, with their average result remaining closest to zero. Most McLaren data points cluster between 0 and -1 positions, showing that drivers generally maintained their qualifying positions rather than losing significant ground during races. There are also several positive outcomes above zero, including races with gains of up to +3 positions. Importantly, McLaren shows fewer extreme negative outliers than the other teams, with only one major loss near -7 positions. Earlier graphs support this trend, as McLaren consistently showed stable tyre degradation, balanced strategy choices, and relatively controlled pit lane performance throughout the season. Together, these results suggest McLaren converted strong qualifying performances into race finishes more consistently than the other teams.

Red Bull recorded the second-best average front-grid conversion, with many results remaining close to the zero line. This means Red Bull drivers often finished near their starting positions when qualifying in the top five. However, unlike McLaren, Red Bull’s results are spread across a wider range, including several races with larger position losses between -3 and -7 places. This suggests Red Bull was capable of maintaining strong results in many races, but their outcomes were less consistent overall. Earlier graphs also showed Red Bull frequently used longer stints and one-stop strategies. While these strategies can help maintain track position, they can also become vulnerable if tyre performance drops later in the race or if safety cars disrupt the pit cycle, which may explain some of the larger negative results shown in the graph.

Ferrari recorded a slightly negative average position change overall. Most Ferrari results cluster between +1 and -2 positions, indicating moderate movement around starting position rather than large gains or losses. However, Ferrari also recorded several larger negative outcomes, including a result near -5 positions. Earlier tyre degradation graphs showed Ferrari experienced relatively high soft-tyre degradation and shorter average stints, which may have reduced their ability to consistently maintain front-grid positions during longer race runs because they are forced to pitstop.

Mercedes recorded the weakest front-grid conversion among the four teams. Although several Mercedes results show positive gains of +2 or +3 positions, the distribution also contains the largest number of significant losses, including multiple races between -4 and -6 positions. This creates the lowest overall average despite some strong recovery drives. Earlier graphs showed Mercedes frequently used aggressive or variable strategies, including the highest proportion of 3+ stop races, which may have contributed to the larger spread of outcomes from strong starting positions.                                    

The graph overall suggests that qualifying near the front did not guarantee stable race results across the season. McLaren showed the most consistent ability to preserve strong starting positions, while Red Bull, Ferrari, and Mercedes experienced larger fluctuations between qualifying and finishing performance. The spread of points also highlights how tyre management, race pace, pit strategy, and race incidents continued to influence outcomes even                                 

The graph overall suggests that qualifying near the front did not guarantee stable race results across the season. McLaren showed the most consistent ability to preserve strong starting positions, while Red Bull, Ferrari, and Mercedes experienced larger fluctuations between qualifying and finishing performance. The spread of points also shows how tyre management, race pace, pit strategy, and race incidents continued to influence outcomes even when teams started near the front of the grid.

# V:  Circuit Performance Characteristics

Straight-line speed and circuit layout are major factors influencing Formula 1 performance. Some circuits reward high top speed and low aerodynamic drag through long straights and heavy braking zones, while others prioritise cornering grip, stability, and car balance through technical sequences of corners. Because each team designs its car with different aerodynamic and mechanical characteristics, performance can vary depending on the type of circuit. A car that performs strongly on straight-heavy tracks may not necessarily achieve the same advantage on more technical circuits, where cornering efficiency and tyre management become more important.

This section examines how Ferrari, McLaren, Mercedes, and Red Bull performed across different circuit characteristics during the 2025 season through two perspectives: overall straight-line speed measured using speed trap data, and average race performance across manually classified corner-heavy and straight-heavy circuits.

In [ ]:
# GRAPH 9: Speed Trap Comparison
import plotly.graph_objects as go
import numpy as np

# Prepare Data
driver_team = drivers_df[['driver_number', 'team_name', 'name_acronym']].drop_duplicates()

speed_df = laps_enriched_df[laps_enriched_df['st_speed'].notna()].merge(
    driver_team, on='driver_number', how='inner'
)
if 'is_pit_out_lap' in speed_df.columns:
    speed_df = speed_df[speed_df['is_pit_out_lap'] != True]

# Filter to top 4
top4_teams = [t for t in speed_df['team_name'].unique()
              if any(k in t for k in ['Ferrari', 'McLaren', 'Mercedes', 'Red Bull'])]
speed_df = speed_df[speed_df['team_name'].isin(top4_teams)]

def short_name(t):
    for k in ['Ferrari', 'McLaren', 'Mercedes', 'Red Bull']:
        if k in t:
            return k
    return t

speed_df['team_short'] = speed_df['team_name'].map(short_name)

# Order by median speed
team_medians = speed_df.groupby('team_short')['st_speed'].median().sort_values(ascending=False)
team_order = team_medians.index.tolist()

# Circuit names for hover
circuit_map = dict(zip(laps_enriched_df['session_key'], laps_enriched_df['circuit_short_name']))
speed_df['circuit'] = speed_df['session_key'].map(circuit_map)

fig = go.Figure()

for i, team in enumerate(team_order):
    team_data = speed_df[speed_df['team_short'] == team]
    color = TEAM_COLORS.get(team, '#FFFFFF')
    median_val = team_data['st_speed'].median()

    jitter = np.random.uniform(-0.25, 0.25, len(team_data))
    y_pos = np.full(len(team_data), i) + jitter

    fig.add_trace(go.Scatter(
        x=team_data['st_speed'].values,
        y=y_pos,
        mode='markers',
        marker=dict(color=color, size=5, opacity=0.4),
        name=team,
        showlegend=False,
        hovertemplate=(
            '<b>%{customdata[0]}</b> — %{customdata[1]}<br>'
            'Lap %{customdata[2]}<br>'
            'Speed: %{x:.0f} km/h<extra></extra>'
        ),
        customdata=list(zip(
            team_data['name_acronym'].values,
            team_data['circuit'].values,
            team_data['lap_number'].values
        ))
    ))

    # Median line
    fig.add_shape(type='line', x0=median_val, x1=median_val,
                  y0=i - 0.35, y1=i + 0.35,
                  line=dict(color='white', width=4))
    fig.add_shape(type='line', x0=median_val, x1=median_val,
                  y0=i - 0.35, y1=i + 0.35,
                  line=dict(color=color, width=2))

    # Median annotation above the line
    fig.add_annotation(
        x=median_val, y=i + 0.38,
        text=f"<b>Median: {median_val:.0f} km/h</b>",
        showarrow=False,
        font=dict(color=color, size=11),
        yanchor='bottom'
    )

    # Team Logo
    if team in TEAM_LOGOS:
        fig.add_layout_image(dict(
            source=TEAM_LOGOS[team],
            x=0.03, y=i,
            xref="paper", yref="y",
            sizex=0.05, sizey=0.6,
            xanchor="center", yanchor="middle",
            layer="above",
        ))

fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text='<b>Season Speed Trap Comparison Across All Circuits</b>',
        font=dict(size=22),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(title='Speed Trap (km/h)', zeroline=False),
    yaxis=dict(
        tickvals=list(range(len(team_order))),
        ticktext=['' for _ in team_order],
        zeroline=False,
        range=[-0.7, len(team_order) - 0.3]
    ),
    margin=dict(l=80),
)

style_axis(fig, '')

fig.show()

This strip plot compares straight-line speed trap values recorded by Ferrari, McLaren, Mercedes, and Red Bull across all circuits during the 2025 season. Each point represents a single speed trap measurement from the OpenF1 lap dataset, while the x-axis shows recorded speed in kilometres per hour. Speed trap values are typically measured at the fastest section of a circuit straight and provide an indication of a car’s top-speed performance. Only laps with valid speed trap data were included in the analysis, and pit-out laps were removed to avoid distorted readings caused by drivers accelerating out of the pit lane rather than completing a normal racing lap. Teams are ordered from highest to lowest median speed. The vertical marker line for each team shows the median speed trap value across the season.

Mercedes recorded the highest median speed trap value at 309 km/h. Their distribution also extends furthest into the highest speed range, with several readings above 340 km/h and peaks approaching 360 km/h. This suggests Mercedes produced the strongest overall straight-line speed across the season. However, the graph also shows a relatively wide spread of values, indicating their top speed performance varied more significantly between circuits and race conditions. Earlier tyre degradation graphs showed Mercedes also performed strongly during long hard-tyre stints, suggesting the team combined strong straight-line speed with stable long-run pace.

Ferrari recorded the second-highest median at 306 km/h. Most Ferrari readings are concentrated between approximately 285 and 325 km/h, showing consistently competitive straight-line speed across a wide range of circuits. Compared to Mercedes, Ferrari produced fewer extremely high peak readings, but their distribution remains dense around the upper speed range. This suggests Ferrari maintained strong and reliable top-speed performances.

Red Bull recorded a median speed of 305 km/h, only slightly below Ferrari. Their distribution is similarly wide, with frequent readings above 330 km/h and several very high peak speeds. However, compared to Ferrari and Mercedes, Red Bull also shows a larger number of lower-speed readings across different circuits. This wider spread suggests Red Bull’s straight-line performance may have varied more depending on track layout, setup configuration, or race conditions. Earlier graphs also showed Red Bull frequently relied on longer stints and one-stop strategies, indicating the team often prioritised overall race management rather than maximising top speed alone.

McLaren recorded the lowest median speed trap value at 304 km/h. Although the difference compared to Ferrari and Red Bull is small, McLaren’s distribution contains fewer extremely high-speed readings above 330 km/h. Most of their values remain concentrated within the central range of the graph, suggesting more stable but slightly lower peak straight-line performance overall. Despite this, earlier graphs showed McLaren maintained some of the most stable tyre degradation patterns and strongest position conversion results, indicating the team compensated through consistent race pace and tyre management rather than relying primarily on maximum straight-line speed.

Overall, the graph shows that the four teams operated within a narrow straight-line performance window, with only a 5 km/h difference separating the highest and lowest median speed trap values. Mercedes produced the strongest peak straight-line speeds overall, Ferrari maintained highly consistent high-speed performance, Red Bull showed larger variation across circuits, and McLaren achieved slightly lower top speeds while remaining competitive through consistency in other performance areas.

In [ ]:
# GRAPH 10: Corner-Heavy vs Straight-Heavy Circuit Performance
import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

CIRCUIT_TYPE = {

    # Corner-Heavy Circuits
    "Hungaroring": "Corner-Heavy",
    "Singapore": "Corner-Heavy",
    "Suzuka": "Corner-Heavy",
    "Zandvoort": "Corner-Heavy",
    "Imola": "Corner-Heavy",
    "Silverstone": "Corner-Heavy",
    "Austin": "Corner-Heavy",
    "Interlagos": "Corner-Heavy",

    # Straight Heavy Circuits
    "Monza": "Straight-Heavy",
    "Jeddah": "Straight-Heavy",
    "Las Vegas": "Straight-Heavy",
    "Spa-Francorchamps": "Straight-Heavy",
    "Baku": "Straight-Heavy",
    "Montreal": "Straight-Heavy",
    "Spielberg": "Straight-Heavy",
    "Mexico City": "Straight-Heavy",
    "Yas Marina Circuit": "Straight-Heavy",
    "Lusail": "Straight-Heavy",
}

available_circuits = laps_enriched_df["circuit_short_name"].dropna().unique()
circuit_to_type = {c: CIRCUIT_TYPE[c] for c in available_circuits if c in CIRCUIT_TYPE}

corner_list = sorted([c for c, t in circuit_to_type.items() if t == "Corner-Heavy"])
straight_list = sorted([c for c, t in circuit_to_type.items() if t == "Straight-Heavy"])

round_type = {}
round_name = {}
for _, row in schedule_df.iterrows():
    rnd = int(row["round"])
    race_name = str(row.get("raceName", ""))
    circuit_id = str(row.get("circuitId", ""))
    locality = str(row.get("locality", ""))
    for cname, ctype in circuit_to_type.items():
        cn_lower = cname.lower()
        if (cn_lower in race_name.lower() or cn_lower in circuit_id.lower() or
            cn_lower in locality.lower() or circuit_id.lower() in cn_lower or
            locality.lower() in cn_lower):
            round_type[rnd] = ctype
            round_name[rnd] = race_name
            break

def match_team(cn):
    cn = str(cn).lower()
    for t in teams_order:
        if t.lower() in cn: return t
    return None

res = results_df.copy()
res["team"] = res["constructorName"].apply(match_team)
res = res[res["team"].isin(teams_order)].copy()
res["position"] = pd.to_numeric(res["position"], errors="coerce")
res = res[res["position"].notna()].copy()
res["position"] = res["position"].astype(int)
res["round"] = res["round"].astype(int)
res = res[res["status"].str.contains(r"Finished|\+", na=False)].copy()
res["circuit_type"] = res["round"].map(round_type)
res = res[res["circuit_type"].notna()].copy()

team_stats = res.groupby(["team", "circuit_type"]).agg(
    avg_pos=("position", "mean"),
    n=("position", "count")
).reset_index()


# Slope
fig = go.Figure()

# Collect Team Data
team_data = {}
for team in teams_order:
    td = team_stats[team_stats["team"] == team]
    cr = td[td["circuit_type"] == "Corner-Heavy"]
    sr = td[td["circuit_type"] == "Straight-Heavy"]
    if cr.empty or sr.empty:
        continue
    color = TEAM_COLORS[team]
    r_c, g_c, b_c = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    team_data[team] = {
        "c": cr["avg_pos"].values[0], "s": sr["avg_pos"].values[0],
        "cn": int(cr["n"].values[0]), "sn": int(sr["n"].values[0]),
        "color": color, "r": r_c, "g": g_c, "b": b_c,
    }

# Background
fig.add_shape(type="rect", x0=-0.3, x1=0.5, y0=0, y1=1,
    xref="x", yref="paper",
    fillcolor="rgba(255,140,0,0.015)", line=dict(width=0), layer="below")
fig.add_shape(type="rect", x0=0.5, x1=1.3, y0=0, y1=1,
    xref="x", yref="paper",
    fillcolor="rgba(100,180,255,0.015)", line=dict(width=0), layer="below")
fig.add_shape(type="line", x0=0.5, x1=0.5, y0=0, y1=1,
    xref="x", yref="paper",
    line=dict(color="rgba(255,255,255,0.05)", width=1, dash="dot"), layer="below")

# Draw teams
for team, d in team_data.items():
    rc, gc, bc = d["r"], d["g"], d["b"]
    col = d["color"]

    # Outer glow
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[d["c"], d["s"]], mode="lines",
        line=dict(color=f"rgba({rc},{gc},{bc},0.04)", width=24),
        showlegend=False, hoverinfo="skip"))
    # Inner glow
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[d["c"], d["s"]], mode="lines",
        line=dict(color=f"rgba({rc},{gc},{bc},0.12)", width=10),
        showlegend=False, hoverinfo="skip"))
    # Core line
    delta = d["s"] - d["c"]
    direction = "worse on straights" if delta > 0 else "better on straights"
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[d["c"], d["s"]], mode="lines",
        line=dict(color=col, width=3.5),
        showlegend=False,
        hovertemplate=(
            f"<b>{team}</b><br>"
            f"Corner-Heavy: P{d['c']:.1f} ({d['cn']} races)<br>"
            f"Straight-Heavy: P{d['s']:.1f} ({d['sn']} races)<br>"
            f"Δ {abs(delta):.1f} — {direction}<extra></extra>")))

    # Left dot
    fig.add_trace(go.Scatter(
        x=[0], y=[d["c"]], mode="markers",
        marker=dict(color=col, size=12, line=dict(color="rgba(0,0,0,0.5)", width=1.5)),
        showlegend=False,
        hovertemplate=f"<b>{team}</b> · Corner-Heavy<br>P{d['c']:.1f} ({d['cn']} races)<extra></extra>"))

    # Right dot
    fig.add_trace(go.Scatter(
        x=[1], y=[d["s"]], mode="markers",
        marker=dict(color=col, size=12, line=dict(color="rgba(0,0,0,0.5)", width=1.5)),
        showlegend=False,
        hovertemplate=f"<b>{team}</b> · Straight-Heavy<br>P{d['s']:.1f} ({d['sn']} races)<extra></extra>"))

# Sort teams
sorted_teams = sorted(team_data.items(), key=lambda x: x[1]["c"])

# Nudge overlapping labels
label_positions = []
min_gap = 0.45
for team, d in sorted_teams:
    y = d["c"]
    if label_positions and y - label_positions[-1] < min_gap:
        y = label_positions[-1] + min_gap
    label_positions.append(y)

for i, (team, d) in enumerate(sorted_teams):
    label_y = label_positions[i]
    delta = d["s"] - d["c"]
    delta_str = f"+{delta:.1f}" if delta > 0 else f"{delta:.1f}"

    fig.add_annotation(
        x=-0.06, y=label_y,
        text=(f"<b>{team}</b>   "
              f"<span style='color:rgba(255,255,255,0.45)'>P{d['c']:.1f} → P{d['s']:.1f}  ({delta_str})</span>"),
        font=dict(color=d["color"], size=11),
        showarrow=False, xanchor="right", yanchor="middle")

# Column headers
fig.add_annotation(x=0.28, y=1.06, xref="paper", yref="paper",
    text="<b>CORNER-HEAVY</b>",
    font=dict(size=14, color="rgba(255,160,40,0.55)"),
    showarrow=False, xanchor="center", yanchor="bottom")
fig.add_annotation(x=0.28, y=1.03, xref="paper", yref="paper",
    text=(" · ".join(corner_list)),
    font=dict(size=7.5, color="rgba(255,255,255,0.25)"),
    showarrow=False, xanchor="center", yanchor="top")

fig.add_annotation(x=0.72, y=1.06, xref="paper", yref="paper",
    text="<b>STRAIGHT-HEAVY</b>",
    font=dict(size=14, color="rgba(100,180,255,0.55)"),
    showarrow=False, xanchor="center", yanchor="bottom")
fig.add_annotation(x=0.72, y=1.03, xref="paper", yref="paper",
    text=(" · ".join(straight_list)),
    font=dict(size=7.5, color="rgba(255,255,255,0.25)"),
    showarrow=False, xanchor="center", yanchor="top")

style = {**GRAPH_STYLE, "height": 650}

fig.update_layout(
    **style,
    title=dict(
        text="<b>Corner-Heavy vs Straight-Heavy Circuit Performance</b>",
        font=dict(size=20, color="#E8E8E8"),
        x=0.5, xanchor="center", y=0.965,
    ),
    xaxis=dict(
        range=[-0.3, 1.3],
        showgrid=False, zeroline=False, showticklabels=False,
    ),
    yaxis=dict(
        title=dict(text="Avg Finish Position", font=dict(size=12, color="#555")),
        autorange="reversed",
        gridcolor="rgba(255,255,255,0.03)", showgrid=True,
        tickfont=dict(size=11, color="#444"),
        dtick=1, tickprefix="P",
    ),
    showlegend=False,
    margin=dict(t=115, b=35, l=185, r=35),
    hoverlabel=dict(
        bgcolor="rgba(15,15,15,0.92)",
        bordercolor="rgba(255,255,255,0.15)",
        font=dict(size=12, color="#E8E8E8", family="Arial"),
    ),
)

fig.show()

This slope chart compares average finishing position for Ferrari, McLaren, Mercedes, and Red Bull across two manually classified circuit categories between corner-heavy circuits and straight-heavy circuits. Corner-heavy circuits were identified as tracks where technical corners and direction changes, while straight-heavy circuits emphasise top speed and straight-line efficiency. The classifications were manually selected by the author after reviewing circuit descriptions and layouts. The average finishing position of each team's driver were caculated for each circuit type.

McLaren recorded the strongest overall results in both circuit categories. Their average finish position changed only slightly from P2.5 on corner-heavy circuits to P2.8 on straight-heavy circuits, a difference of just +0.3 positions. This was the smallest performance change among all four teams, suggesting McLaren maintained the most balanced car across different circuit styles. Earlier graphs also support this pattern, as McLaren consistently showed stable tyre degradation, controlled pit stop performance, and strong front-grid position conversion throughout the season. The small difference between the two circuit categories suggests McLaren’s competitiveness depended less on track layout than the other teams. The car performed equally well on corner-heavy circuits and straight-heavy circuits.

Mercedes showed the largest improvement between the two circuit types. Their average finish improved from P6.3 on corner-heavy circuits to P5.3 on straight-heavy circuits, an average gain of 1 position. This aligns with the earlier speed trap graph, where Mercedes recorded the highest median straight-line speed at 309 km/h. The data suggests Mercedes performed more strongly on circuits where straight-line speed and top-speed efficiency played a larger role. Earlier degradation graphs also showed Mercedes performed particularly well during long hard-tyre stints, which may have further benefited them on high-speed circuits with longer race runs.

Red Bull also improved on straight-heavy circuits, moving from an average finish of P6.1 to P5.7. Although the improvement of 0.4 positions is smaller than Mercedes, the trend still suggests Red Bull performed slightly better on circuits rewarding straight-line speed. Earlier graphs also showed Red Bull recorded strong overall speed trap values and frequently maintained stable pace across longer tyre stints. Together, these trends suggest Red Bull remained competitive on faster circuits where maintaining speed over long runs was important.

Ferrari showed almost no difference between the two categories, changing only from P5.8 on corner-heavy circuits to P5.9 on straight-heavy circuits. This suggests Ferrari’s overall race performance remained relatively consistent regardless of circuit type. Unlike Mercedes and Red Bull, Ferrari does not show a clear preference toward straight-heavy tracks despite recording strong overall speed trap performance earlier in the analysis. Instead, the graph suggests Ferrari’s results were influenced more by overall race execution and consistency than by any major circuit-type advantage.

Overall, McLaren remained the most balanced across both track types, Mercedes benefited most from straight-heavy circuits, Red Bull showed moderate improvement on higher-speed tracks, and Ferrari remained relatively unchanged between the two categories. The relatively small gaps overall also suggest that differences in race execution, tyre management, and qualifying performance likely remained just as or more important than circuit suitability across the season.

# VI: Limitations & Future Works

While the project provides useful insights into team performance, tyre behaviour, pit strategy, and race execution during the 2025 Formula 1 season, several limitations should be acknowledged. Formula 1 is an extremely complex sport where performance is influenced by many interconnected variables, and not every factor could be fully measured within the scope of this investigation.

One major limitation is that many graphs rely on simplified indicators. For example, the tyre degradation analysis measured changes in lap time over a stint as a proxy for tyre wear rather than measuring actual tyre surface temperature, tyre compound chemistry, ground temperature, and more. Lap times can also be affected by fuel load, traffic, dirty air, safety cars, ERS deployment, weather conditions, track evolution, and driver management, meaning tyre degradation cannot be isolated perfectly from other race influences.

Several analyses also depended on publicly available datasets from OpenF1 and Ergast rather than official FIA telemetry. While these datasets are highly useful, they still contain limitations in precision and completeness. Certain metrics, such as exact pit stop reaction timing, tyre temperatures, brake temperatures, DRS activation efficiency, fuel flow, aerodynamic balance, or real-time engine modes, were not available. As a result, some conclusions must be interpreted as performance trends rather than exact measurements Driver-specific factors such as driving style, tyre management ability, overtaking skill, qualifying performance, and consistency were also not isolated separately from overall team performance.

The classification of circuits into “corner-heavy” and “straight-heavy” categories also involved a level of subjective interpretation. Track descriptions and layouts were researched using external Formula 1 circuit analyses, and circuits were manually grouped based on their overall characteristics. However, many modern circuits contain both long straights and technical corner sections, meaning some tracks cannot be perfectly represented by only one category.

Another limitation is that averages were frequently used throughout the investigation. While averages help identify long-term trends, they can sometimes hide race-specific context. For example, a team may record strong average tyre degradation despite struggling heavily at a small number of circuits. Similarly, pit stop performance or position-change metrics may be influenced by unusual races involving safety cars, weather, red flags, collisions, or mechanical failures.

The project also focused only on the top four teams. This allowed comparisons to remain clearer and more manageable, but it excluded midfield and lower-ranked teams that may have demonstrated different strategic approaches or performance characteristics.

If the investigation were expanded further, several additional areas could be explored. One major improvement would be analysing aerodynamic performance more directly. This could include studying drag levels, slip streams, dirty air, downforce efficiency, DRS effectiveness, sector-by-sector performance, and more. More advanced telemetry data could also allow comparisons between acceleration profiles, throttle application, braking zones, and energy recovery systems.

Future investigations could also examine the technical design of each car in greater detail. This may include comparing engine manufacturers, power unit performance, suspension concepts, wheelbase length, rear wing configurations, floor designs, cooling systems, and how different aerodynamic setups influenced race pace at different circuits. Fuel usage and hybrid system deployment could also be analysed to better understand how teams balanced straight-line speed against tyre management and race efficiency. One of the main limitations of this investigation is that it does not directly analyse the engineering and aerodynamic differences between each car, despite these factors being major determinants of Formula 1 performance.

Machine learning or predictive modelling could also be introduced in future work. For example, race outcomes, tyre degradation, or pit stop strategies could potentially be predicted using historical telemetry and race data. Sector-level analysis and weather-adjusted models may also improve the accuracy of performance comparisons across different circuits.

This project will be maintained by the author and any additional changes, and extra/related works will be uploaded to the Github Repository linked in the introductary paragraph.

# VII: Conclusion

This investigation analysed the performance of Ferrari, McLaren, Mercedes, and Red Bull during the 2025 Formula 1 season using race results, lap timing data, tyre performance metrics, pit stop analysis, and circuit-type comparisons. By combining multiple datasets from OpenF1 and Ergast, the project explored how different teams approached tyre management, pit strategy, race execution, position conversion, and overall performance across different track characteristics.

The investigation identified clear differences between the four teams. McLaren consistently demonstrated the most stable overall performance across multiple areas of the analysis. Their tyre degradation curves remained controlled across all compounds, their pit strategies were balanced and low-risk, and they converted strong qualifying positions into consistent race finishes with relatively little variation. Mercedes showed strong long-run pace, particularly on harder compounds, while also achieving the largest average position gains after pit stops. Ferrari displayed competitive peak performance in several areas, including straight-line speed and qualifying conversion, but also showed larger variability in tyre degradation and race outcomes across certain graphs. Red Bull frequently relied on longer stints and one-stop strategies while maintaining strong straight-line performance, although some graphs showed greater variation in race finishing consistency compared to McLaren.

The investigation also demonstrated how closely connected different areas of Formula 1 performance are. Tyre degradation influenced pit stop strategy, pit strategy affected track position, and circuit characteristics influenced how effectively teams could maximise their car strengths. Rather than relying on a single performance metric, the project showed that successful Formula 1 performance depends on balancing speed, consistency, tyre management, strategic decision-making, and adaptability across different race conditions.

Although several limitations existed due to data availability and the complexity of Formula 1, the investigation successfully identified meaningful trends across the 2025 season.

Overall, the investigation demonstrates that Formula 1 performance cannot be explained by one factor alone. Success across a season depends on the interaction between car design, tyre behaviour, race strategy, circuit characteristics, and driver execution. By analysing these factors together, the project provides a broader understanding of how the top Formula 1 teams achieved performance across the 2025 championship season.

The author would also like to acknowledge and thank Dr. Emanuel Santos, Ms. Hien Le, and Duc Minh Do for their guidance, feedback, and support throughout the development of this investigation.